#pytti: python text to image


This is a closed beta. Leak it if you must, information wants to be free.

pytti is made possible by supporters like you. [Thank you.](https://www.youtube.com/watch?v=TexDW6nEhgU)

Original Notebook by H. R. (@sportsracer48)
Based on techniques by @advadnoun and Katherine Crawson (@rivershavewings)

Local Support (2024) by [Bardia323](https://github.com/Bardia323)

Jan 2026 Log:
```
- New Clip models support added by Chris May
```

Jan 2024 Log:
```
1. forked GMA and taming-transformers and made ad hoc patches to prevent error on local systems.
2. added missing dependencies for local use.
3. changed a directory name in GMA from utils -> utilus as well as associated files to avoid name clashes with Adabins utils.
4. commented out tqdm_color_scheme as it was throwing error on local systems.
```

Oct 2024 Log:
```
1. Added smarter encoding (10x faster video styling speed)
2. Tensor operations optimizations
```

In [ ]:
# @title Licensed under the MIT License
# Copyleft (c) 2021 Henry Rachootin

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THE SOFTWARE.

In [ ]:

#@title Help { form-width: "30px" }
`scenes:` Descriptions of scenes you want generated, separated by `||`. Each scene can contain multiple prompts, separated by `|`.
*Example:* `Winter sunrise | icy landscape || Winter day | snowy skyline || Winter sunset | chilly air || Winter night | clear sky` would go through several winter scenes.
**Advanced:** weight prompts with `description:weight`. Higher `weight` values will be prioritized by the optimizer, and negative `weight` values will remove the description from the image. The default weight is $1$. Weights can also be functions of $t$ to change over the course of an animation.
*Example scene:* `blue sky:10|martian landscape|red sky:-1` would try to turn the martian sky blue.
**Advanced:** stop prompts once the image matches them sufficiently with `description:weight:stop`. `stop` should be between $0$ and $1$ for positive prompts, or between $-1$ and $0$ for negative prompts. Lower `stop` values will have more effect on the image (remember that $-1<-0.5<0$). A prompt with a negative `weight` will often go haywire without a stop. Stops can also be functions of $t$ to change over the course of an animation.
*Example scene:* `Feathered dinosaurs|birds:1:0.87|scales:-1:-.9|text:-1:-.9` Would try to make feathered dinosaurs, lightly like birds, without scales or text, but without making 'anti-scales' or 'anti-text.'

#**NEW:**

**Advanced:** Use `description:weight_mask description` with a text prompt as `mask`. The prompt will only be applied to areas of the image that match `mask description` according to CLIP.
*Example scene:* `Khaleesi Daenerys Targaryen | mother of dragons | dragon:3_baby` would only apply the weight `dragon` to parts of the image that match `baby`, thus turning the babies that `mother` tends to make into dragons (hopefully).
**Advanced:** Use `description:weight_[mask]` with a URL or path to an image, or a path to a .mp4 video to use as a `mask`. The prompt will only be applied to the masked (white) areas of the mask image. Use `description:weight_[-mask]` to apply the prompt to the black areas instead.
*Example scene:* `sunlight:3_[mask.mp4]|midnight:3_[-mask.mp4]` Would apply `sunlight` in the white areas of `mask.mp4`, and `midnight` in the black areas.
**Legacy:** Directional weights will still work as before, but they aren't as good as masks.
**Advanced:** Use `[path or url]` as a prompt to add a semantic image prompt. This will be read by CLIP and understood as a near perfect text description of the image.
*Example scene:* `[artist signature.png]:-1:-.95|[https://i.redd.it/ewpeykozy7e71.png]:3|fractal clouds|hole in the sky`

---

`scene_prefix:` text prepended to the beginning of each scene.
*Example:* `Trending on Arstation|`
`scene_suffix:` text appended to the end of each scene.
*Example:* ` by James Gurney`
`interpolation_steps:` number of steps to spend smoothly transitioning from the last scene at the start of each scene. $200$ is a good default. Set to $0$ to disable.
`steps_per_scene:` total number of steps to spend rendering each scene. Should be at least `interpolation_steps`. This will indirectly control the total length of an animation.
---
#**NEW**:
`direct_image_prompts:` paths or urls of images that you want your image to look like in a literal sense, along with `weight_mask` and `stop` values, separated by `|`.
Apply masks to direct image prompts with `path or url of image:weight_path or url of mask` For video masks it must be a path to an mp4 file.
**Legacy** latent image prompts are no more. They are now rolled into direct image prompts.
---
`init_image:` path or url of start image. Works well for creating a central focus.
`direct_init_weight:` Defaults to $0$. Use the initial image as a direct image prompt. Equivalent to adding `init_image:direct_init_weight` as a `direct_image_prompt`. Supports weights, masks, and stops.
`semantic_init_weight:` Defaults to $0$. Defaults to $0$. Use the initial image as a semantic image prompt. Equivalent to adding `[init_image]:direct_init_weight` as a prompt to each scene in `scenes`. Supports weights, masks, and stops. **IMPORTANT** since this is a semantic prompt, you still need to put the mask in `[` `]` to denote it as a path or url, otherwise it will be read as text instead of a file.
---

`width`, `height:` image size. Set one of these $-1$ to derive it from the aspect ratio of the init image.
`pixel_size:` integer image scale factor. Makes the image bigger. Set to $1$ for VQGAN or face VRAM issues.
`smoothing_weight:` makes the image smoother. Defaults to $0$ (no smoothing). Can also be negative for that deep fried look.
`image_model:` select how your image will be represented.
`vqgan_model:` select your VQGAN version (only for `image_model: VQGAN`)
`random_initial_palette:` if checked, palettes will start out with random colors. Otherwise they will start out as grayscale. (only for `image_model: Limited Palette`)
`palette_size:` number of colors in each palette. (only for `image_model: Limited Palette`)
`palettes:` total number of palettes. The image will have `palette_size*palettes` colors total. (only for `image_model: Limited Palette`)
`gamma:` relative gamma value. Higher values make the image darker and higher contrast, lower values make the image lighter and lower contrast. (only for `image_model: Limited Palette`). $1$ is a good default.
`hdr_weight:` how strongly the optimizer will maintain the `gamma`. Set to $0$ to disable. (only for `image_model: Limited Palette`)
`palette_normalization_weight:` how strongly the optimizer will maintain the palettes' presence in the image. Prevents the image from losing palettes. (only for `image_model: Limited Palette`)
`show_palette:` check this box to see the palette each time the image is displayed. (only for `image_model: Limited Palette`)
`target_pallete:` path or url of an image which the model will use to make the palette it uses.
`lock_pallete:` force the model to use the initial palette (most useful from restore, but will force a grayscale image or a wonky palette otherwise).

---

`animation_mode:` select animation mode or disable animation.
`sampling_mode:` how pixels are sampled during animation. `nearest` will keep the image sharp, but may look bad. `bilinear` will smooth the image out, and `bicubic` is untested :)
`infill_mode:` select how new pixels should be filled if they come in from the edge.
* mirror: reflect image over boundary
* wrap: pull pixels from opposite side
* black: fill with black
* smear: sample closest pixel in image
`pre_animation_steps:` number of steps to run before animation starts, to begin with a stable image. $250$ is a good default.
`steps_per_frame:` number of steps between each image move. $50$ is a good default.
`frames_per_second:` number of frames to render each second. Controls how $t$ is scaled.
`direct_stabilization_weight: ` keeps the current frame as a direct image prompt. For `Video Source` this will use the current frame of the video as a direct image prompt. For `2D` and `3D` this will use the shifted version of the previous frame. Also supports masks: `weight_mask.mp4`.
`semantic_stabilization_weight: ` keeps the current frame as a semantic image prompt. For `Video Source` this will use the current frame of the video as a direct image prompt. For `2D` and `3D` this will use the shifted version of the previous frame. Also supports masks: `weight_[mask.mp4]` or `weight_mask phrase`.
`depth_stabilization_weight: ` keeps the depth model output somewhat consistent at a *VERY* steep performance cost. For `Video Source` this will use the current frame of the video as a semantic image prompt. For `2D` and `3D` this will use the shifted version of the previous frame. Also supports masks: `weight_mask.mp4`.
`edge_stabilization_weight: ` keeps the images contours somewhat consistent at very little performance cost. For `Video Source` this will use the current frame of the video as a direct image prompt with a sobel filter. For `2D` and `3D` this will use the shifted version of the previous frame. Also supports masks: `weight_mask.mp4`.
`flow_stabilization_weight: ` used for `animation_mode: 3D` and `Video Source` to prevent flickering. Comes with a slight performance cost for `Video Source`, and a great one for `3D`, due to implementation differences. Also supports masks: `weight_mask.mp4`. For video source, the mask should select the part of the frame you want to move, and the rest will be treated as a still background.
---
`video_path: ` path to mp4 file for `Video Source`
`frame_stride` advance this many frames in the video for each output frame. This is surprisingly useful. Set to $1$ to render each frame. Video masks will also step at this rate.
`reencode_each_frame: ` check this box to use each video frame as an `init_image` instead of warping each output frame into the init for the next. Cuts will still be detected and trigger a reencode.
`flow_long_term_samples: ` Sample multiple frames into the past for consistent interpolation even with disocclusion, as described by [Manuel Ruder, Alexey Dosovitskiy, and Thomas Brox (2016)](https://arxiv.org/abs/1604.08610). Each sample is twice as far back in the past as the last, so the earliest sampled frame is $2^{\text{long_term_flow_samples}}$ frames in the past. Set to $0$ to disable.
---

`translate_x:` horizontal image motion as a function of time $t$ in seconds.
`translate_y:` vertical image motion as a function of time $t$ in seconds.
`translate_z_3d:` forward image motion as a function of time $t$ in seconds. (only for `animation_mode:3D`)
`rotate_3d:` image rotation as a quaternion $\left[r,x,y,z\right]$ as a function of time $t$ in seconds. (only for `animation_mode:3D`)
`rotate_2d:` image rotation in degrees as a function of time $t$ in seconds. (only for `animation_mode:2D`)
`zoom_x_2d:` horizontal image zoom as a function of time $t$ in seconds. (only for `animation_mode:2D`)
`zoom_y_2d:` vertical image zoom as a function of time $t$ in seconds. (only for `animation_mode:2D`)
`lock_camera:` check this box to prevent all scrolling or drifting. Makes for more stable 3D rotations. (only for `animation_mode:3D`)
`field_of_view:` vertical field of view in degrees. (only for `animation_mode:3D`)
`near_plane:` closest depth distance in pixels. (only for `animation_mode:3D`)
`far_plane:` farthest depth distance in pixels. (only for `animation_mode:3D`)
---

`file_namespace:` output directory name.
`allow_overwrite:` check to overwrite existing files in `file_namespace`.
`display_every:` how many steps between each time the image is displayed in the notebook.
`clear_every:` how many steps between each time notebook console is cleared.
`display_scale:` image display scale in notebook. $1$ will show the image at full size. Does not affect saved images.
`save_every:` how many steps between each time the image is saved. Set to `steps_per_frame` for consistent animation.
`backups:` number of backups to keep (only the oldest backups are deleted). Large images make very large backups, so be warned. Set to `all` to save all backups. These are used for the `flow_long_term_samples` so be sure that this is at least $2^{\text{flow_long_term_samples}}+1$ for `Video Source` mode.
`show_graphs:` check this to see graphs of the loss values each time the image is displayed. Disable this for local runtimes.
`approximate_vram_usage:` currently broken. Don't believe its lies.

---

`ViTB32, ViTB16, RN50, RN50x4:` select your CLIP models. These take a lot of VRAM.
`learning_rate:` how quickly the image changes.
`reset_lr_each_frame:` the optimizer will adaptively change the learning rate, so this will thwart it.
`seed:` pseudorandom seed.
---

`cutouts:` number of cutouts. Reduce this to use less VRAM at the cost of quality and speed.
`cut_pow:` should be positive. Large values shrink cutouts, making the image more detailed, small values expand the cutouts, making it more coherent. $1$ is a good default. $3$ or higher can cause crashes.
`cutout_border:` should be between $0$ and $1$. Allows cutouts to poke out over the edges of the image by this fraction of the image size, allowing better detail around the edges of the image. Set to $0$ to disable. $0.25$ is a good default.
`border_mode:` how to fill cutouts that stick out over the edge of the image. Match with `infill_mode` for consistent infill.
* clamp: move cutouts back onto image
* mirror: reflect image over boundary
* wrap: pull pixels from opposite side
* black: fill with black
* smear: sample closest pixel in image

#Step 1: Setup
Run the cells in this section once for each runtime, or after a factory reset.

In [ ]:
#@title 1.0 NVIDIA-SMI (optional)
#@markdown View information about your runtime GPU.
#@markdown Google will connect you to an industrial strength GPU, which is needed to run
#@markdown this notebook. You can also disable error checking on your GPU to get some
#@markdown more VRAM, at a marginal cost to stability. You will have to restart the runtime after
#@markdown disabling it.
enable_error_checking = False#@param {type:"boolean"}
if enable_error_checking:
  !nvidia-smi
else:
  !nvidia-smi
  !nvidia-smi -i 0 -e 0

In [ ]:
#@title 1.1 Use Colab
#@markdown Check this button if using Colab.
#@
use_colab = True #@param{type:"boolean"}

In [ ]:
#@title 1.15 Mount google drive (optional) - Only works with Colab
#@markdown Mounting your drive is optional but recommended. You can even restore from google randomly
#@markdown kicking you out if you mount your drive.
mount_drive = False #@param{type:"boolean"}
if mount_drive:
  from google.colab import drive
  drive.mount('/content/drive', force_remount = True)
  !mkdir -p /content/drive/MyDrive/pytti_test
  %cd /content/drive/MyDrive/pytti_test
else:
  print("Initiating local mode.")

In [ ]:
#@title 1.2 Install everything else
#@markdown Run this cell on a fresh runtime to install the libraries and modules.
import os
import shutil
from os.path import exists as path_exists
if path_exists('/content/drive/MyDrive/pytti_test'):
  %cd /content/drive/MyDrive/pytti_test
else:
  %cd /content

try:
  from adjustText import adjust_text
  import pytti, torch
  everything_installed = True
except ModuleNotFoundError:
  everything_installed = False
def install_everything():
  !pip install bunch==1.0.0
  !pip install pandas==1.5.3
  !pip install opencv-python
  !pip install ninja
  !pip install scipy
  !pip install imageio
  !pip install seaborn
  !pip install transformers
  !pip install PyGLM
  !pip install ftfy regex tqdm omegaconf pytorch-lightning
  !pip install kornia
  !pip install einops
  !pip install imageio-ffmpeg
  !pip install adjustText exrex bunch
  !pip install matplotlib-label-lines
  !pip install --upgrade gast
  !pip install gdown
  !pip install open_clip_torch
  !git clone https://github.com/openai/CLIP.git
  !git clone https://github.com/bardia323/taming-transformers.git
  if not path_exists('./pytti'):
    !git clone --branch p5 https://github.com/bardia323/pytti.git
  else:
      pass
    #!rm -r pytti
    #!git clone --branch p5 https://github.com/sportsracer48/pytti.git
  !git clone https://github.com/shariqfarooq123/AdaBins.git
  # --- MiDaS / DPT depth backend (alternative to AdaBins, see cell 2.15) ---
  !pip install timm
  !git clone https://github.com/isl-org/MiDaS.git
  !git clone https://github.com/bardia323/GMA.git
  if use_colab:
    !mkdir -p AdaBins/pretrained
  else:
    !mkdir AdaBins\pretrained
  if (not path_exists('AdaBins/pretrained/AdaBins_nyu.pt')) and (not path_exists('AdaBins_nyu.pt')):
    try:
      !gdown "1lvyZZbC9NLcS8a__YPcUP7rDiIpbRpoF"
    except:
      !gdown "1zgGJrkFkJbRouqMaWArXE4WF_rhj-pxW"
    if mount_drive:
        src_path = '/content/AdaBins_nyu.pt'
        dst_path = os.path.join('content','AdaBins', 'pretrained', 'AdaBins_nyu.pt')
    else:
        src_path = 'AdaBins_nyu.pt'
        dst_path = os.path.join('AdaBins', 'pretrained', 'AdaBins_nyu.pt')
        shutil.move(src_path, dst_path)


  """
  if use_colab:
    !move AdaBins_nyu.pt AdaBins/pretrained/AdaBins_nyu.pt
  else:
    !move AdaBins_nyu.pt AdaBins\pretrained\AdaBins_nyu.pt
  """


  from pytti.Notebook import change_tqdm_color
  #change_tqdm_color()
  !mkdir images_out
  !mkdir videos

force_install = False #@param{type:"boolean"}
if not everything_installed or force_install:
  install_everything()
elif everything_installed:
  #from pytti.Notebook import change_tqdm_color
  #change_tqdm_color()
  pass

# Step 2: Run it!
Edit the parameters, or load saved parameters, then run the model.

In [ ]:
#@title #2.1 Parameters:
#@markdown ---
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from os.path import exists as path_exists
if path_exists('/content/drive/MyDrive/pytti_test'):
  %cd /content/drive/MyDrive/pytti_test
  drive_mounted = True
else:
  drive_mounted = False
try:
  from pytti.Notebook import  get_last_file
  #from pytti.Notebook import change_tqdm_color
except ModuleNotFoundError:
  if drive_mounted:
    #THIS IS NOT AN ERROR. This is the code that would
    #make an error if something were wrong.
    raise RuntimeError('ERROR: please run setup (step 1.3).')
  else:
    #THIS IS NOT AN ERROR. This is the code that would
    #make an error if something were wrong.
    raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1.3).')

#change_tqdm_color()

import glob, json, random, re, math
try:
  from bunch import Bunch
except ModuleNotFoundError:
  if drive_mounted:
    #THIS IS NOT AN ERROR. This is the code that would
    #make an error if something were wrong.
    raise RuntimeError('ERROR: please run setup (step 1.3).')
  else:
    #THIS IS NOT AN ERROR. This is the code that would
    #make an error if something were wrong.
    raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1.3).')

#these are used to make the defaults look pretty
model_default = None
random_seed = None
all  = math.inf
derive_from_init_aspect_ratio = -1

def define_parameters():
  locals_before = locals().copy()
  #@markdown ###Prompts:

  scenes = "rolling sunlit countryside with a golden wheat field rippling in the wind, thick painterly brushstrokes, a tall dark cypress tree standing in the center, scattered green shrubs and twisting plants across the hillside, distant blue hills under a wide sky with swirling clouds, expressive color and motion || a vibrant coastal cliff village with white houses and red roofs perched above a turquoise sea, boats tied to a wooden dock, tall grasses and wildflowers moving in the wind, dramatic clouds and radiant sunlight reflecting across the wave || a small white farmhouse with a yellow metal roof beside stone steps and a vegetable garden, sunflower plants and around the house, a clothesline in the wind, bright clouds and glowing sky bands behind distant hills " #@param{type:"string"}
  #scenes = convert_string_multiple(scene, 20)
  scene_prefix = "vibrant psychedelic post-impressionist oil painting of "#@param{type:"string"}
  scene_suffix = "| in the style of Van Gogh | text:-1:-0.95 | text:-1"#@param{type:"string"}
  interpolation_steps = 550#@param{type:"number"}
  steps_per_scene =  4500#@param{type:"raw"}
  #@markdown ---
  #@markdown ###Image Prompts:
  direct_image_prompts   = ""#@param{type:"string"}
  #@markdown ---
  #@markdown ###Initial image:
  init_image = "/home/administrator/pytti/inits/VField.jpg"#@param{type:"string"}
  direct_init_weight =  ""#@param{type:"string"}
  semantic_init_weight = ""#@param{type:"string"}
  #@markdown ---
  #@markdown ###Image:
  #@markdown Use `image_model` to select how the model will encode the image
  image_model = "Limited Palette" #@param ["VQGAN", "Limited Palette", "Unlimited Palette", "MultiRes(DAS)", "MultiResLimitedPalette", "GNCA"]
  #@markdown image_model | description | strengths | weaknesses
  #@markdown --- | -- | -- | --
  #@markdown  VQGAN | classic VQGAN image | smooth images | limited datasets, slow, VRAM intesnsive
  #@markdown  Limited Palette | pytti differentiable palette | fast,  VRAM scales with `palettes` | pixel images
  #@markdown  Unlimited Palette | simple RGB optimization | fast, VRAM efficient | pixel images
  #@markdown The output image resolution will be `width` $\times$ `pixel_size` by height $\times$ `pixel_size` pixels.
  #@markdown The easiest way to run out of VRAM is to select `image_model` VQGAN without reducing
  #@markdown `pixel_size` to $1$.

  #@markdown For `animation_mode: 3D` the minimum resoultion is about 450 by 400 pixels.
  width =  1920#@param {type:"raw"}
  height =  1080#@param {type:"raw"}
  pixel_size = 1#@param{type:"number"}
  smoothing_weight =  2.7#@param{type:"number"}
  #@markdown `VQGAN` specific settings:
  vqgan_model = "imagenet" #@param ["imagenet", "coco", "wikiart", "sflckr", "openimages"]
  #@markdown `Limited Palette` specific settings:
  random_initial_palette = True#@param{type:"boolean"}
  palette_size = 64#@param{type:"number"}
  palettes   = 32#@param{type:"number"}
  gamma = 2.2#@param{type:"number"}
  hdr_weight = 1#@param{type:"number"}
  palette_normalization_weight = 1#@param{type:"number"}
  show_palette = False #@param{type:"boolean"}
  target_palette = ""#@param{type:"string"}
  lock_palette = False #@param{type:"boolean"}

  #@markdown ---

  #@markdown ###Animation:
  animation_mode = "3D" #@param ["off","2D", "3D", "Video Source"]
  sampling_mode = "bicubic" #@param ["bilinear","nearest","bicubic"]
  infill_mode = "smear" #@param ["mirror","wrap","black","smear"]
  pre_animation_steps =  0#@param{type:"number"}
  steps_per_frame =  20#@param{type:"number"}
  frames_per_second =  12#@param{type:"number"}
  #@markdown ---
  #@markdown ###Stabilization Weights:
  direct_stabilization_weight = ""#@param{type:"string"}
  semantic_stabilization_weight = ""#@param{type:"string"}
  depth_stabilization_weight = ""#@param{type:"string"}
  edge_stabilization_weight = ""#@param{type:"string"}
  #@markdown `flow_stabilization_weight` is used for `animation_mode: 3D` and `Video Source`
  flow_stabilization_weight = ""#@param{type:"string"}
  #@markdown ---
  #@markdown ###Video Tracking:
  #@markdown Only for `animation_mode: Video Source`.
  video_path = "/home/administrator/stablewarp/init_images/Hawk_Clip.mp4"#@param{type:"string"}
  frame_stride = 1#@param{type:"number"}
  reencode_each_frame = False #@param{type:"boolean"}
  flow_long_term_samples = 3#@param{type:"number"}
  #@markdown ---
  #@markdown ###Image Motion:
  translate_x    = "0" #@param{type:"string"}
  translate_y    = "0" #@param{type:"string"}
  #@markdown `..._3d` is only used in 3D mode.
  translate_z_3d = "58" #@param{type:"string"}
  #@markdown `rotate_3d` *must* be a `[w,x,y,z]` rotation (unit) quaternion. Use `rotate_3d: [1,0,0,0]` for no rotation.
  #@markdown [Learn more about rotation quaternions here](https://eater.net/quaternions).
  rotate_3d      = "[1,0.002,0,0]"#@param{type:"string"}
  #@markdown `..._2d` is only used in 2D mode.
  rotate_2d      = "0" #@param{type:"string"}
  zoom_x_2d      = "0" #@param{type:"string"}
  zoom_y_2d      = "0" #@param{type:"string"}
  #@markdown  3D camera (only used in 3D mode):
  lock_camera   = False#@param{type:"boolean"}
  field_of_view = 75#@param{type:"number"}
  near_plane    = 1#@param{type:"number"}
  far_plane     = 10000#@param{type:"number"}

  #@markdown ---
  #@markdown ###Output:
  file_namespace = "VincentDay"#@param{type:"string"}
  if file_namespace == '':
    file_namespace = 'out'
  allow_overwrite = True#@param{type:"boolean"}
  base_name = file_namespace
  if not allow_overwrite and path_exists(f'images_out/{file_namespace}'):
    _, i = get_last_file(f'images_out/{file_namespace}',
                         f'^(?P<pre>{re.escape(file_namespace)}\\(?)(?P<index>\\d*)(?P<post>\\)?_1\\.png)$')
    if i == 0:
      print(f"WARNING: file_namespace {file_namespace} already has images from run 0")
    elif i is not None:
      print(f"WARNING: file_namespace {file_namespace} already has images from runs 0 through {i}")
  elif glob.glob(f'images_out/{file_namespace}/{base_name}_*.png'):
    print(f"WARNING: file_namespace {file_namespace} has images which will be overwritten")
  try:
    del i
    del _
  except NameError:
    pass
  del base_name
  display_every = steps_per_frame #@param{type:"raw"}
  clear_every = 0 #@param{type:"raw"}
  display_scale = 0.5#@param{type:"number"}
  save_every = steps_per_frame #@param{type:"raw"}
  backups =  2**(flow_long_term_samples+1)+1 #this is used for video transfer, so don't lower it if that's what you're doing#@param {type:"raw"}
  show_graphs = False #@param{type:"boolean"}
  approximate_vram_usage = False#@param{type:"boolean"}

  #@markdown ---
  #@markdown ###Classic Models (2022):
  #@markdown Quality settings from Dribnet's CLIPIT (https://github.com/dribnet/clipit).
  #@markdown Selecting too many will use up all your VRAM and slow down the model.
  #@markdown I usually use ViTB32, ViTB16, and RN50 if I get a A100, otherwise I just use ViT32B.

  #@markdown quality | CLIP models
  #@markdown --- | --
  #@markdown  draft | ViTB32
  #@markdown  normal | ViTB32, ViTB16
  #@markdown  high | ViTB32, ViTB16, RN50
  #@markdown  best | ViTB32, ViTB16, RN50x4

  #@markdown ###New Models (2026)
  #@markdown CLIP model | Description
  #@markdown --- | --
  #@markdown  ConvNeXtLarge | like the RN50's, good for texture and detail
  #@markdown  #DFN_ViT_H | coherence and photorealism
  #@markdown  #MetaCLIP_ViT_B | supposedly better at coherence than VitB32 and uses the same VRAM

  ViTB32 = True #@param{type:"boolean"}
  ViTB16 = False #@param{type:"boolean"}
  RN50 = False #@param{type:"boolean"}
  RN50x4 = False #@param{type:"boolean"}

  ViTL14 = False #@param{type:"boolean"}
  ConvNeXtLarge = False #@param{type:"boolean"}
  DFN_ViT_H = False #@param{type:"boolean"}
  MetaCLIP_ViT_B = True #@param{type:"boolean"}
  #ViTH14_336 = False #@param{type:"boolean"}
  #@markdown the default learning rate is `0.1` for all the VQGAN models
  #@markdown except openimages, which is `0.15`. For the palette modes the
  #@markdown default is `0.02`.
  learning_rate =  0.02#@param{type:"raw"}
  reset_lr_each_frame = False#@param{type:"boolean"}
  seed =  3546464225445#@param{type:"raw"}
  #@markdown **Cutouts**:

  #@markdown [Cutouts are how CLIP sees the image.](https://twitter.com/remi_durant/status/1460607677801897990)
  cutouts =  42#@param{type:"number"}
  cut_pow =  1.78#@param {type:"number"}
  cutout_border =  0.25#@param {type:"number"}
  #@markdown NOTE: prompt masks (`promt:weight_[mask.png]`) will not work right on '`wrap`' or '`mirror`' mode.
  border_mode = "smear" #@param ["clamp","mirror","wrap","black","smear"]

  if seed is None:
    seed = random.randint(-0x8000_0000_0000_0000, 0xffff_ffff_ffff_ffff)
  locals_after = locals().copy()
  for k in locals_before.keys():
    del locals_after[k]
  del locals_after['locals_before']
  return locals_after

params = Bunch(define_parameters())
print("SETTINGS:")
print(json.dumps(params))

In [ ]:
#@title 2.2 Load settings (optional)
#@markdown copy the `SETTINGS:` output from the **Parameters** cell (tripple click to select the whole
#@markdown line from `{'scenes'...` to `}`) and paste them in a note to save them for later.

#@markdown Paste them here in the future to load those settings again. Running this cell with blank settings won't do anything.
load_settings = False #@param{type:"boolean"}
if load_settings:
  from os.path import exists as path_exists
  if path_exists('/content/drive/MyDrive/pytti_test'):
    %cd /content/drive/MyDrive/pytti_test
    drive_mounted = True
  else:
    drive_mounted = False
  try:
    from pytti.Notebook import *
  except ModuleNotFoundError:
    if drive_mounted:
      #THIS IS NOT AN ERROR. This is the code that would
      #make an error if something were wrong.
      raise RuntimeError('ERROR: please run setup (step 1.3).')
    else:
      #THIS IS NOT AN ERROR. This is the code that would
      #make an error if something were wrong.
      raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1.3).')
  change_tqdm_color()

  import json, random
  try:
    from bunch import Bunch
  except ModuleNotFoundError:
    if drive_mounted:
      #THIS IS NOT AN ERROR. This is the code that would
      #make an error if something were wrong.
      raise RuntimeError('ERROR: please run setup (step 1.3).')
    else:
      #THIS IS NOT AN ERROR. This is the code that would
      #make an error if something were wrong.
      raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1.3).')

  settings = "{\"scenes\": \" A landscape with a black hole as the sun in retro scifi art style| text:-1:-0.95 |  writing:-1:-0.95 |  portrait:-1:-0.95 | Scifi art | Psychedelic | clouds:-1:-0.95\", \"scene_prefix\": \"\", \"scene_suffix\": \"\", \"interpolation_steps\": 1500, \"steps_per_scene\": 40000, \"direct_image_prompts\": \"/content/Blackhole(4)_57.png:1000\", \"init_image\": \"\", \"direct_init_weight\": \"\", \"semantic_init_weight\": \"\", \"image_model\": \"Limited Palette\", \"width\": 900, \"height\": 1600, \"pixel_size\": 1, \"smoothing_weight\": 2.3, \"vqgan_model\": \"wikiart\", \"random_initial_palette\": true, \"palette_size\": 32, \"palettes\": 8, \"gamma\": 4, \"hdr_weight\": 2, \"palette_normalization_weight\": 1, \"show_palette\": false, \"target_palette\": \"\", \"lock_palette\": false, \"animation_mode\": \"3D\", \"sampling_mode\": \"bicubic\", \"infill_mode\": \"smear\", \"pre_animation_steps\": 500, \"steps_per_frame\": 40, \"frames_per_second\": 12, \"direct_stabilization_weight\": \"\", \"semantic_stabilization_weight\": \"\", \"depth_stabilization_weight\": \"\", \"edge_stabilization_weight\": \"\", \"flow_stabilization_weight\": \"\", \"video_path\": \"\", \"frame_stride\": 1, \"reencode_each_frame\": false, \"flow_long_term_samples\": 1, \"translate_x\": \"0\", \"translate_y\": \"0\", \"translate_z_3d\": \"0\", \"rotate_3d\": \"[1,0,0,0]\", \"rotate_2d\": \"0\", \"zoom_x_2d\": \"0\", \"zoom_y_2d\": \"0\", \"lock_camera\": false, \"field_of_view\": 95, \"near_plane\": 1, \"far_plane\": 10000, \"file_namespace\": \"Blackhole\", \"allow_overwrite\": false, \"display_every\": 40, \"clear_every\": 0, \"display_scale\": 0.5, \"save_every\": 40, \"backups\": 5, \"show_graphs\": false, \"approximate_vram_usage\": false, \"ViTB32\": true, \"ViTB16\": true, \"RN50\": true, \"RN50x4\": false, \"learning_rate\": null, \"reset_lr_each_frame\": false, \"seed\": -1207693824359482134, \"cutouts\": 40, \"cut_pow\": 1.75, \"cutout_border\": 0.25, \"border_mode\": \"clamp\"}"#@param{type:"string"}
  #@markdown Check `random_seed` to overwrite the seed from the settings with a random one for some variation.
  random_seed = False #@param{type:"boolean"}

  if settings != '':
    params = load_settings(settings, random_seed)

In [ ]:
#@title 2.15  Optimizations & CLIP-space sampling  { form-width: "320px" }
#@markdown Runtime patches over the p5 `pytti` classes. **Every toggle defaults to legacy
#@markdown behaviour** and each patch is wrapped in try/except, so the notebook still runs
#@markdown exactly as before unless you flip something on. Run this AFTER **2.1 Parameters**
#@markdown and BEFORE **2.3 Run it!**. Re-run it any time you change a toggle.

import torch, math, traceback
import torch.nn.functional as F
import pandas as pd

#@markdown #### Speed
enable_tf32           = True   #@param{type:"boolean"}
enable_amp            = True   #@param{type:"boolean"}
enable_torch_compile  = False  #@param{type:"boolean"}
profile_steps         = False  #@param{type:"boolean"}
profile_every         = 20     #@param{type:"number"}
fix_dataframe_cpu     = True    #@param{type:"boolean"}
sanitize_grads        = True   #@param{type:"boolean"}
#@markdown #### CLIP sampling / quality
antialiased_cutouts   = True    #@param{type:"boolean"}
vectorized_cutouts    = True    #@param{type:"boolean"}
lite_augs             = False   #@param{type:"boolean"}
slerp_interpolation   = True    #@param{type:"boolean"}
directional_clip      = False   #@param{type:"boolean"}
directional_source    = "a photo"  #@param{type:"string"}
#@markdown #### Stochastic sampling (Langevin / SGLD)
sgld_sampling         = False   #@param{type:"boolean"}
sgld_noise            = 0.003   #@param{type:"number"}
#@markdown #### Depth backend for 3D motion (EXPERIMENTAL)
use_midas_depth       = False   #@param{type:"boolean"}
midas_model           = "DPT_Hybrid"  #@param ["MiDaS_small","DPT_Hybrid","DPT_Large"]
midas_invert          = True    #@param{type:"boolean"}

# Mutable config object that the patched methods read at call-time
OPT = dict(amp=enable_amp, compile=enable_torch_compile, df=fix_dataframe_cpu,
           sanitize=sanitize_grads,
           prof=profile_steps, prof_every=int(profile_every),
           aa=antialiased_cutouts, vec=vectorized_cutouts, lite=lite_augs, slerp=slerp_interpolation,
           directional=directional_clip, src_text=directional_source,
           sgld=sgld_sampling, sgld_noise=float(sgld_noise))
import builtins; builtins.OPT = OPT

# ---------- global backends ----------
if enable_tf32:
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        print("[ok] TF32 + cudnn.benchmark on")
    except Exception:
        traceback.print_exc()

import pytti
from pytti import DEVICE, format_input, replace_grad, cat_with_pad
from pytti.ImageGuide import DirectImageGuide
from pytti.Perceptor.Embedder import HDMultiClipEmbedder
import pytti.Perceptor.Prompt as _PromptMod
from pytti.Perceptor.Prompt import Prompt, spherical_dist_loss

# ===========================================================================
# 1) Anti-aliased cutouts  (bicubic+antialias instead of adaptive_avg_pool2d)
# ===========================================================================
try:
    if not hasattr(HDMultiClipEmbedder, "_orig_make_cutouts"):
        HDMultiClipEmbedder._orig_make_cutouts = HDMultiClipEmbedder.make_cutouts
    def _embed_augs(self):
        # default-off lighter augmentation stack -- RandomPerspective/RandomErasing/ColorJitter
        # have the heaviest (fp32) backward; flip and affine keep most of the robustness benefit.
        if not OPT.get("lite", False):
            return self.augs
        if getattr(self, "_lite_augs", None) is None:
            import kornia.augmentation as K
            from torch import nn as _nn
            self._lite_augs = _nn.Sequential(
                K.RandomHorizontalFlip(p=0.3),
                K.RandomAffine(degrees=15, translate=0.1, p=0.7, padding_mode="border"),
                _nn.Identity())
        return self._lite_augs

    def _make_cutouts_vec(self, input, side_x, side_y, cut_size, device=DEVICE):
        # Vectorized extraction: build all `cutn` crops with ONE affine_grid + grid_sample. The
        # fused grid_sample has a cheaper FORWARD *and* BACKWARD than a Python loop of per-cutout
        # antialiased resizes. Crop size/offset distribution matches legacy.
        cutn = self.cutn
        max_size = min(side_x, side_y)
        min_frac = cut_size / max_size
        paddingx = min(round(side_x * self.padding), side_x)
        paddingy = min(round(side_y * self.padding), side_y)
        H_in, W_in = input.shape[-2:]
        C = input.shape[1]

        r = torch.empty(cutn, device=device).normal_(mean=.8, std=.3).clip_(min_frac, 1.)
        size = (max_size * r.pow(self.cut_pow)).floor().clamp_(min=1.)
        rx = torch.rand(cutn, device=device); ry = torch.rand(cutn, device=device)
        offx_max = side_x - size + 1
        offy_max = side_y - size + 1
        if self.border_mode == "clamp":
            offx = (rx * (offx_max + 2 * paddingx) - paddingx).floor().clamp_(min=0)
            offy = (ry * (offy_max + 2 * paddingy) - paddingy).floor().clamp_(min=0)
            offx = torch.minimum(offx, offx_max); offy = torch.minimum(offy, offy_max)
            x0, y0 = offx, offy
        else:
            px = torch.clamp(size, max=float(paddingx)); py = torch.clamp(size, max=float(paddingy))
            offx = (rx * (offx_max + 2 * px) - px).floor()
            offy = (ry * (offy_max + 2 * py) - py).floor()
            x0, y0 = paddingx + offx, paddingy + offy

        theta = torch.zeros(cutn, 2, 3, device=device, dtype=torch.float32)
        theta[:, 0, 0] = size / W_in
        theta[:, 1, 1] = size / H_in
        theta[:, 0, 2] = (x0 + size / 2) / W_in * 2 - 1
        theta[:, 1, 2] = (y0 + size / 2) / H_in * 2 - 1
        grid = F.affine_grid(theta, (cutn, C, cut_size, cut_size), align_corners=False).to(input.dtype)
        src = input.expand(cutn, -1, -1, -1)
        pad_mode = "reflection" if self.border_mode == "mirror" else "border"
        cutouts = F.grid_sample(src, grid, mode="bilinear", padding_mode=pad_mode, align_corners=False)

        cutouts = _embed_augs(self)(cutouts)
        if self.noise_fac:
            facs = cutouts.new_empty([cutn, 1, 1, 1]).uniform_(0, self.noise_fac)
            cutouts.add_(facs * torch.randn_like(cutouts))

        offsets = torch.stack([offx / side_x, offy / side_y], dim=1)
        sizes   = torch.stack([size / side_x, size / side_y], dim=1)
        return cutouts, offsets, sizes

    def _make_cutouts(self, input, side_x, side_y, cut_size, device=DEVICE):
        if OPT.get("vec", False):
            return _make_cutouts_vec(self, input, side_x, side_y, cut_size, device=device)
        if not OPT["aa"]:
            return HDMultiClipEmbedder._orig_make_cutouts(self, input, side_x, side_y, cut_size, device=device)
        max_size = min(side_x, side_y)
        paddingx = min(round(side_x * self.padding), side_x)
        paddingy = min(round(side_y * self.padding), side_y)
        cutouts, offsets, sizes = [], [], []
        for _ in range(self.cutn):
            size = int(max_size * (torch.zeros(1,).normal_(mean=.8, std=.3)
                                   .clip(cut_size/max_size, 1.) ** self.cut_pow))
            offsetx_max = side_x - size + 1
            offsety_max = side_y - size + 1
            if self.border_mode == "clamp":
                offsetx = torch.clamp((torch.rand([])*(offsetx_max+2*paddingx) - paddingx).floor().int(), 0, offsetx_max)
                offsety = torch.clamp((torch.rand([])*(offsety_max+2*paddingy) - paddingy).floor().int(), 0, offsety_max)
                cutout = input[:, :, offsety:offsety+size, offsetx:offsetx+size]
            else:
                px = min(size, paddingx); py = min(size, paddingy)
                offsetx = (torch.rand([])*(offsetx_max+2*px) - px).floor().int()
                offsety = (torch.rand([])*(offsety_max+2*py) - py).floor().int()
                cutout = input[:, :, paddingy+offsety:paddingy+offsety+size, paddingx+offsetx:paddingx+offsetx+size]
            # the only change vs legacy: band-limited resample -> cleaner CLIP gradients
            cutout = F.interpolate(cutout, size=(cut_size, cut_size), mode="bicubic",
                                   align_corners=False, antialias=True)
            cutouts.append(cutout)
            offsets.append(torch.as_tensor([[offsetx/side_x, offsety/side_y]]).to(device))
            sizes.append(torch.as_tensor([[size/side_x, size/side_y]]).to(device))
        cutouts = self.augs(torch.cat(cutouts))
        offsets = torch.cat(offsets); sizes = torch.cat(sizes)
        if self.noise_fac:
            facs = cutouts.new_empty([self.cutn, 1, 1, 1]).uniform_(0, self.noise_fac)
            cutouts.add_(facs * torch.randn_like(cutouts))
        return cutouts, offsets, sizes
    HDMultiClipEmbedder.make_cutouts = _make_cutouts
    print("[ok] cutouts patch installed (vec:", OPT.get("vec", False), "| aa:", OPT["aa"], ")")
except Exception:
    traceback.print_exc()

# ===========================================================================
# 2) Lazy torch.compile of the CLIP perceptors (first forward traces, then fast)
# ===========================================================================
try:
    if not hasattr(HDMultiClipEmbedder, "_orig_forward"):
        HDMultiClipEmbedder._orig_forward = HDMultiClipEmbedder.forward
    def _embed_forward(self, diff_image, input=None, device=DEVICE):
        if OPT["compile"] and not getattr(self, "_pytti_compiled", False):
            import torch._dynamo as _dynamo
            # CLIP stacks many ResidualAttentionBlock instances that share ONE `forward`;
            # dynamo guards on each instance's id and blows the default cache (8) -> endless
            # recompiles. Raise the limits so every block traces & caches exactly once.
            _dynamo.config.cache_size_limit = max(getattr(_dynamo.config, "cache_size_limit", 8), 256)
            if hasattr(_dynamo.config, "accumulated_cache_size_limit"):
                _dynamo.config.accumulated_cache_size_limit = max(_dynamo.config.accumulated_cache_size_limit, 1024)
            for p in self.perceptors:
                try:
                    p.encode_image = torch.compile(p.encode_image, fullgraph=False)  # default mode (no CUDA graphs -> autograd-safe)
                except Exception as e:
                    print("torch.compile failed on a perceptor:", e)
            self._pytti_compiled = True
            print("[ok] perceptors compiled (first iterations will be slow while tracing)")
        return HDMultiClipEmbedder._orig_forward(self, diff_image, input=input, device=device)
    HDMultiClipEmbedder.forward = _embed_forward
except Exception:
    traceback.print_exc()

# ===========================================================================
# 3) Directional CLIP loss (StyleGAN-NADA style text-difference anchoring)
# ===========================================================================
try:
    _PromptMod._SRC_EMB = None
    @torch.no_grad()
    def _get_src_emb():
        if _PromptMod._SRC_EMB is None:
            from CLIP import clip
            percs = pytti.Perceptor.CLIP_PERCEPTORS
            _PromptMod._SRC_EMB = cat_with_pad(
                [p.encode_text(clip.tokenize(OPT["src_text"]).to(DEVICE)).float() for p in percs]).detach()
        return _PromptMod._SRC_EMB
    if not hasattr(Prompt, "_orig_forward"):
        Prompt._orig_forward = Prompt.forward
    def _prompt_forward(self, embed, position, size, offset=0.0, device=DEVICE):
        # only applies to plain text prompts; image prompts use the legacy path
        if (not OPT["directional"]) or (type(self) is not Prompt) or (not hasattr(self, "embeds")):
            return Prompt._orig_forward(self, embed, position, size, offset=offset, device=device)
        if (not self.enabled) or self.weight in ["0", 0]:
            return torch.as_tensor(offset, device=device), offset
        src = _get_src_emb()
        # compare the (image - source) direction to the (text - source) direction
        dists_raw = spherical_dist_loss(embed - src, self.embeds - src) + offset
        weight = torch.as_tensor(pytti.parametric_eval(self.weight), device=device)
        stop   = torch.as_tensor(pytti.parametric_eval(self.stop),   device=device)
        mask_stops, mask_weights = self.mask(position, size, embed.detach())
        weight = torch.as_tensor(mask_weights, device=device) * weight
        sign_offset = weight.sign().clamp(max=0)
        dists = dists_raw * weight.sign()
        stops = torch.maximum(mask_stops + sign_offset, stop)
        dists = weight.abs() * replace_grad(dists, torch.maximum(dists, stops))
        return dists.mean(), dists_raw.mean()
    Prompt.forward = _prompt_forward
    print("[ok] directional-CLIP patch installed (toggle:", OPT["directional"], ")")
except Exception:
    traceback.print_exc()

# ===========================================================================
# 4) Patched training step: AMP (CLIP fp16), SLERP interp, SGLD noise, CPU df fix
# ===========================================================================
try:
    if not hasattr(DirectImageGuide, "_orig_clear_dataframe"):
        DirectImageGuide._orig_clear_dataframe = DirectImageGuide.clear_dataframe
    def _clear_dataframe(self):
        self._rec = []; self._idx = []; self._rec_t = []; self._rec_names = []
        DirectImageGuide._orig_clear_dataframe(self)
    DirectImageGuide.clear_dataframe = _clear_dataframe

    def _flush_loss_records(self):
        # Materialise the pending window of GPU loss scalars to Python floats with a
        # SINGLE host sync (falls back to per-row if logged keys changed mid-window,
        # e.g. across a scene boundary).
        pending = getattr(self, "_rec_t", None)
        if not pending:
            return
        if not hasattr(self, "_rec"): self._rec = []
        try:
            mat = torch.stack(pending).cpu().tolist()          # one .cpu() for the whole window
            for nm, row in zip(self._rec_names, mat):
                self._rec.append(dict(zip(nm, row)))
        except Exception:
            for nm, t in zip(self._rec_names, pending):
                self._rec.append(dict(zip(nm, t.detach().cpu().tolist())))
        self._rec_t = []; self._rec_names = []
        self.dataframe = [pd.DataFrame(self._rec, index=self._idx[:len(self._rec)])]
    DirectImageGuide._flush_loss_records = _flush_loss_records

    def _train(self, i, prompts, interp_prompts, loss_augs, interp_steps=0, save_loss=True):
        self.optimizer.zero_grad()
        _prof = OPT.get("prof", False) and torch.cuda.is_available()
        if _prof:
            _pe = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
            _pe[0].record()
        z = self.image_rep.decode_training_tensor()
        if _prof: _pe[1].record()

        use_amp = OPT["amp"] and torch.cuda.is_available()
        if self.embedder is not None:
            # only the CLIP forward runs in fp16; embeds are .float()'d so losses stay fp32
            with torch.cuda.amp.autocast(enabled=use_amp):
                image_embeds, offsets, sizes = self.embedder(self.image_rep, input=z)
            all_prompts = prompts + interp_prompts
            formatted_inputs = {}
            for prompt in set(all_prompts):
                formatted_inputs[prompt] = {
                    "embeds":  format_input(image_embeds, self.embedder, prompt),
                    "offsets": format_input(offsets, self.embedder, prompt),
                    "sizes":   format_input(sizes, self.embedder, prompt)}
        else:
            formatted_inputs = {}

        formatted_z = {aug: format_input(z, self.image_rep, aug) for aug in loss_augs}

        # interpolation: legacy linear crossfade, or equal-power "slerp" crossfade
        if i < interp_steps:
            tt = i / interp_steps
            if OPT["slerp"]:
                w_new, w_old = math.sin(tt*math.pi/2), math.cos(tt*math.pi/2)
            else:
                w_new, w_old = tt, 1 - tt
            interp_losses = [prompt(formatted_inputs[prompt]["embeds"],
                                    formatted_inputs[prompt]["offsets"],
                                    formatted_inputs[prompt]["sizes"])[0] * w_old
                             for prompt in interp_prompts if prompt in formatted_inputs]
        else:
            w_new = 1; interp_losses = [0]

        prompt_losses = {}
        for prompt in prompts:
            if prompt not in formatted_inputs:        # no embedder (e.g. NCA with CLIP off) -> skip CLIP
                continue
            loss = prompt(formatted_inputs[prompt]["embeds"],
                          formatted_inputs[prompt]["offsets"],
                          formatted_inputs[prompt]["sizes"])
            loss[0].mul_(w_new)
            prompt_losses[prompt] = loss

        aug_losses = {aug: aug(formatted_z[aug], self.image_rep) for aug in loss_augs}
        image_losses = {aug: aug(self.image_rep) for aug in self.image_rep.image_loss()}

        total_loss = sum(l[0] for l in prompt_losses.values()) + \
                     sum(l[0] for l in aug_losses.values()) + \
                     sum(l[0] for l in image_losses.values()) + \
                     sum(interp_losses)
        if _prof: _pe[2].record()

        if save_loss:
            # Collect component scalars as DETACHED GPU tensors -- no host<->device sync
            # per step. They are materialised to Python floats in batches at flush time,
            # turning ~(1 + n_prompts + n_augs) syncs/step into one sync per flush window.
            names = ["TOTAL"]
            tl = total_loss if torch.is_tensor(total_loss) else torch.as_tensor(float(total_loss), device=DEVICE)
            vals = [tl.detach().float().mean()]
            for k, v in prompt_losses.items(): names.append(str(k)); vals.append(v[0].detach().float().mean())
            for k, v in aug_losses.items():    names.append(str(k)); vals.append(v[0].detach().float().mean())
            for k, v in image_losses.items():  names.append(str(k)); vals.append(v[0].detach().float().mean())
            if OPT["df"]:
                if not hasattr(self, "_rec_t"):
                    self._rec_t = []; self._rec_names = []; self._idx = []
                    if not hasattr(self, "_rec"): self._rec = []
                self._rec_t.append(torch.stack(vals)); self._rec_names.append(names); self._idx.append(i)
                if len(self._rec_t) >= 10:
                    self._flush_loss_records()
            else:
                loss_dict = {n: float(t) for n, t in zip(names, vals)}
                if not self.dataframe:
                    self.dataframe = [pd.DataFrame(loss_dict, index=[i])]
                else:
                    self.dataframe[0] = pd.concat([self.dataframe[0], pd.DataFrame(loss_dict, index=[i])])

        total_loss.backward()
        if _prof: _pe[3].record()

        if OPT["sgld"] and OPT["sgld_noise"] > 0:
            with torch.no_grad():
                for p in self.image_rep.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * OPT["sgld_noise"])

        if OPT.get("sanitize", True):
            with torch.no_grad():
                for _p in self.image_rep.parameters():
                    if _p.grad is not None:
                        torch.nan_to_num_(_p.grad, nan=0.0, posinf=0.0, neginf=0.0)

        self.optimizer.step()
        self.image_rep.update()
        if _prof:
            _pe[4].record()
            if not hasattr(self, "_prof_acc"):
                self._prof_acc = [0.0, 0.0, 0.0, 0.0]; self._prof_n = 0
            torch.cuda.synchronize()                       # the only added sync, prof-mode only
            self._prof_acc[0] += _pe[0].elapsed_time(_pe[1])   # image decode
            self._prof_acc[1] += _pe[1].elapsed_time(_pe[2])   # CLIP forward + loss
            self._prof_acc[2] += _pe[2].elapsed_time(_pe[3])   # backward
            self._prof_acc[3] += _pe[3].elapsed_time(_pe[4])   # optimizer step + image update
            self._prof_n += 1
            if self._prof_n >= int(OPT.get("prof_every", 20)):
                a = [x / self._prof_n for x in self._prof_acc]; tot = sum(a)
                print(f"[prof] decode {a[0]:5.1f} | clip+loss {a[1]:6.1f} | backward {a[2]:6.1f} | "
                      f"step+update {a[3]:5.1f} | total {tot:6.1f} ms/it ({1000.0/max(tot,1e-6):4.1f} it/s)")
                self._prof_acc = [0.0, 0.0, 0.0, 0.0]; self._prof_n = 0
        return {"TOTAL": float(total_loss)}
    DirectImageGuide.train = _train
    print("[ok] training-step patch installed  (amp:", OPT["amp"],
          "| slerp:", OPT["slerp"], "| sgld:", OPT["sgld"], "| df_fix:", OPT["df"], ")")
except Exception:
    traceback.print_exc()

# ===========================================================================
# 5) MiDaS depth backend instead of AdaBins (EXPERIMENTAL)
# ===========================================================================
try:
    import numpy as np, sys, importlib.util
    from pytti.LossAug.DepthLoss import DepthLoss      # live class (shared with Transforms.zoom_3d)
    _DLmod = sys.modules['pytti.LossAug.DepthLoss']

    # save pristine originals once; recover from a fresh module load in case they were
    # already overwritten earlier this session (so toggling MiDaS off truly restores AdaBins)
    if not hasattr(DepthLoss, "_orig_get_depth"):
        _spec = importlib.util.find_spec("pytti.LossAug.DepthLoss")
        _fresh = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_fresh)
        DepthLoss._orig_get_depth         = _fresh.DepthLoss.get_depth
        DepthLoss._orig_compute_depth_map = _fresh.DepthLoss.compute_depth_map
        _DLmod._orig_init_AdaBins         = _fresh.init_AdaBins

    if use_midas_depth:
        _midas = {"model": None, "tf": None}
        def _load_midas():
            if _midas["model"] is None:
                m = torch.hub.load("intel-isl/MiDaS", midas_model).to(DEVICE).eval()
                tfs = torch.hub.load("intel-isl/MiDaS", "transforms")
                _midas["model"] = m
                _midas["tf"] = tfs.dpt_transform if "DPT" in midas_model else tfs.small_transform
            return _midas["model"], _midas["tf"]
        # AdaBins-NYU metric range (looked up: min_depth=1e-3, max_depth=10);
        # zoom_3d does np.interp(depth,(1e-3,10),(near*px,far*px)) so values carry over
        ADABINS_MIN_DEPTH, ADABINS_MAX_DEPTH = 1e-3, 10.0
        def _robust_range(x, lo=0.02, hi=0.98):
            f = x.flatten().float()
            if f.numel() > 500000:
                f = f[torch.randint(0, f.numel(), (500000,), device=f.device)]
            return torch.quantile(f, lo), torch.quantile(f, hi)
        def _to_depth(disp):
            # MiDaS/DPT output is disparity (inverse depth): larger = closer.
            disp = disp.float()
            lo, hi = _robust_range(disp)
            disp = ((disp - lo) / (hi - lo + 1e-8)).clamp(0, 1)        # [0,1], 1 = closest
            depth = 1.0 / (disp * (1.0 - 1e-3) + 1e-3)                 # disparity -> metric-like depth
            dlo, dhi = _robust_range(depth)
            depth = ((depth - dlo) / (dhi - dlo + 1e-8)).clamp(0, 1)   # [0,1], 1 = farthest
            if not midas_invert:                                       # escape hatch if a model's sign differs
                depth = 1.0 - depth
            return depth * (ADABINS_MAX_DEPTH - ADABINS_MIN_DEPTH) + ADABINS_MIN_DEPTH
        @torch.no_grad()
        def _get_depth(pil_image):
            model, tf = _load_midas()
            arr = np.array(pil_image.convert("RGB"))
            pred = model(tf(arr).to(DEVICE))
            pred = F.interpolate(pred.unsqueeze(1), size=arr.shape[:2],
                                 mode="bicubic", align_corners=False).squeeze()
            return _to_depth(pred).cpu().numpy(), False
        @torch.no_grad()
        def _compute_depth_map(tensor_input):
            model, _ = _load_midas()
            x = F.interpolate(tensor_input.to(DEVICE), size=(384, 384), mode="bicubic", align_corners=False)
            mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
            std  = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
            pred = model((x - mean) / std)
            return _to_depth(pred).unsqueeze(0).unsqueeze(0)
        DepthLoss.get_depth         = staticmethod(_get_depth)
        DepthLoss.compute_depth_map = staticmethod(_compute_depth_map)
        _DLmod.init_AdaBins         = lambda *a, **k: None            # skip AdaBins load
        print("[ok] MiDaS depth backend active:", midas_model,
              "- EXPERIMENTAL. If 3D pushes the wrong way, flip midas_invert.")
    else:
        # restore AdaBins (no-op the first time if never patched)
        DepthLoss.get_depth         = staticmethod(DepthLoss._orig_get_depth)
        DepthLoss.compute_depth_map = staticmethod(DepthLoss._orig_compute_depth_map)
        _DLmod.init_AdaBins         = _DLmod._orig_init_AdaBins
        print("[ok] depth backend: AdaBins (MiDaS off)")
except Exception:
    traceback.print_exc()
    print("Depth backend setup failed.")

print("\nActive experiments:", {k: v for k, v in OPT.items() if v not in (False, 0, 0.0)})


In [ ]:
#@title 2.16  New Motion: Fractal init + Perlin turbulence
#@markdown - **Fractal (fBm) init** replaces the flat random start for *Limited Palette* with
#@markdown   multi-octave value noise - more natural structure, faster to converge.
#@markdown - **Perlin turbulence** exposes `turb(t, scale, octaves)` and `perlin(x)` inside the
#@markdown   *Image Motion* params (2.1). e.g. `translate_x = "40*turb(t,0.4)"` for organic drift.
#@markdown Run AFTER **2.1 Parameters**, BEFORE **2.3 Run it!**.

import torch, math, traceback
import torch.nn.functional as F
import numpy as np
import pytti
from pytti.Image.PixelImage import PixelImage

fractal_init        = True   #@param{type:"boolean"}
fractal_octaves     = 5      #@param{type:"number"}
fractal_persistence = 0.55   #@param{type:"number"}

MOT = dict(fractal=fractal_init, octaves=int(fractal_octaves), persist=float(fractal_persistence))
import builtins; builtins.MOT = MOT

def _fbm(h, w, octaves, persistence):
    """multi-octave value noise in [0,1], shape (h,w)"""
    img = torch.zeros(1, 1, h, w); amp = 1.0; norm = 0.0
    for o in range(int(octaves)):
        g = max(2, 2 ** (o + 1))
        grid = torch.rand(1, 1, g, g)
        img = img + amp * F.interpolate(grid, size=(h, w), mode="bicubic", align_corners=False)
        norm += amp; amp *= persistence
    img = img / max(norm, 1e-8)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img.squeeze(0).squeeze(0)

try:
    if not hasattr(PixelImage, "_orig_encode_random"):
        PixelImage._orig_encode_random = PixelImage.encode_random
    @torch.no_grad()
    def _encode_random(self, random_pallet=False):
        if not MOT["fractal"]:
            return PixelImage._orig_encode_random(self, random_pallet=random_pallet)
        h, w = self.value.shape
        self.value.copy_(_fbm(h, w, MOT["octaves"], MOT["persist"]).to(self.value.device))
        n = self.tensor.shape[0]
        sel = torch.stack([_fbm(h, w, max(2, MOT["octaves"] - 2), MOT["persist"]) for _ in range(n)])
        self.tensor.copy_(sel.to(self.tensor.device))
        if random_pallet:
            self.pallet.uniform_(to=self.pallet_inertia)
        print("[ok] fractal (fBm) init applied")
    PixelImage.encode_random = _encode_random
    print("[ok] fractal-init patch installed (toggle:", MOT["fractal"], ")")
except Exception:
    traceback.print_exc()

# ---- Perlin helpers injected into the parametric-eval namespace ----
try:
    _rng = np.random.default_rng(0)
    _GRAD = _rng.uniform(-1.0, 1.0, 1024)
    def _fade(t): return t * t * t * (t * (t * 6 - 15) + 10)
    def _lerp(a, b, t): return a + t * (b - a)
    def perlin(x):
        """1-D gradient noise, ~[-1,1]"""
        xi = int(math.floor(x)); xf = x - xi; u = _fade(xf)
        g0 = _GRAD[xi % 1024]; g1 = _GRAD[(xi + 1) % 1024]
        return _lerp(g0 * xf, g1 * (xf - 1), u)
    def turb(t, scale=1.0, octaves=4):
        s = 0.0; amp = 1.0; f = scale; norm = 0.0
        for _ in range(int(octaves)):
            s += amp * perlin(t * f); norm += amp; amp *= 0.5; f *= 2
        return s / (norm if norm else 1.0)
    if pytti.math_env is None:
        pytti.parametric_eval("0")        # force math_env to build
    pytti.math_env.update({"turb": turb, "perlin": perlin})
    print("[ok] motion params can now use turb(t,scale,octaves) and perlin(x)")
except Exception:
    traceback.print_exc()


In [ ]:
#@title 2.17  Distill-lineage parameterization tricks (Limited Palette)
#@markdown Image-parameterization experiments in the spirit of distill.pub's
#@markdown "Differentiable Image Parameterizations" / "Feature Visualization".
#@markdown All default OFF; each is isolated. Run AFTER **2.1**, BEFORE **2.3**.

import torch, math, traceback
import torch.nn.functional as F
from pytti import replace_grad
from pytti.Image.PixelImage import PixelImage, break_tensor

gumbel_palette         = False  #@param{type:"boolean"}
gumbel_tau_start       = 1.5    #@param{type:"number"}
gumbel_tau_end         = 0.25   #@param{type:"number"}
gumbel_anneal_steps    = 1500   #@param{type:"number"}
soft_clamp_value       = False  #@param{type:"boolean"}
spectral_precondition  = False  #@param{type:"boolean"}
spectral_alpha         = 1.0    #@param{type:"number"}
laplacian_precondition = False  #@param{type:"boolean"}
#@markdown #### Depth-guided preconditioning (uses the depth model's shapes to shape the gradient)
depth_guided_precondition = False  #@param{type:"boolean"}
depth_guided_mode      = "continuous"  #@param ["continuous", "layers"]
depth_guided_radius    = 8      #@param{type:"number"}
depth_guided_eps       = 0.01   #@param{type:"number"}
depth_guided_layers    = 8      #@param{type:"number"}
depth_guided_refresh   = 20     #@param{type:"number"}
depth_guided_warmup    = 50     #@param{type:"number"}

DIS = dict(gumbel=gumbel_palette, tau0=float(gumbel_tau_start), tau1=float(gumbel_tau_end),
           anneal=int(gumbel_anneal_steps), soft=soft_clamp_value,
           spectral=spectral_precondition, alpha=float(spectral_alpha),
           laplacian=laplacian_precondition,
           depth_guided=depth_guided_precondition, dg_mode=depth_guided_mode,
           dg_radius=int(depth_guided_radius), dg_eps=float(depth_guided_eps),
           dg_layers=int(depth_guided_layers), dg_refresh=int(depth_guided_refresh),
           dg_warmup=int(depth_guided_warmup))
import builtins; builtins.DIS = DIS

# ---- (a) Gumbel-softmax palette selection (annealed temp) + (b) soft-clamp value ----
try:
    if not hasattr(PixelImage, "_orig_decode_tensor"):
        PixelImage._orig_decode_tensor = PixelImage.decode_tensor
    def _decode_tensor(self):
        if not (DIS["gumbel"] or DIS["soft"]):
            return PixelImage._orig_decode_tensor(self)
        width, height = self.image_shape
        pallet = self.sort_pallet()
        gpallet = pallet if self.current_gamma == 1.0 else pallet.pow(self.current_gamma)
        raw_v = self.value
        v = (0.5 * (torch.tanh(4 * (raw_v - 0.5)) + 1)) if DIS["soft"] else raw_v.clamp(0, 1)
        values = v * (self.pallet_size - 1)
        vf, vc, vr, vfr = break_tensor(values)
        vfr = vfr.unsqueeze(-1).unsqueeze(-1)
        pw = self.tensor.movedim(0, 2)
        pallets = F.one_hot(pw.argmax(dim=2), num_classes=self.n_pallets).float().unsqueeze(-1)
        if DIS["gumbel"]:
            step = getattr(self, "_gstep", 0); self._gstep = step + 1
            frac = min(1.0, step / max(1, DIS["anneal"]))
            tau = DIS["tau0"] + (DIS["tau1"] - DIS["tau0"]) * frac
            soft = F.gumbel_softmax(pw, tau=max(1e-3, tau), hard=False, dim=2)
        else:
            soft = F.softmax(pw, dim=2)
        soft = soft.unsqueeze(-1)
        colors_disc = (gpallet[vr] * pallets).sum(dim=2)
        colors_disc = F.interpolate(colors_disc.permute(2, 0, 1).unsqueeze(0).contiguous(),
                                    size=(height, width), mode="nearest")
        colors_cont = gpallet[vf] * (1 - vfr) + gpallet[vc] * vfr
        colors_cont = (colors_cont * soft).sum(dim=2)
        colors_cont = F.interpolate(colors_cont.permute(2, 0, 1).unsqueeze(0).contiguous(),
                                    size=(height, width), mode="nearest")
        return replace_grad(colors_disc, colors_cont * 0.5 + colors_disc * 0.5)
    PixelImage.decode_tensor = _decode_tensor

    if not hasattr(PixelImage, "_orig_update"):
        PixelImage._orig_update = PixelImage.update
    @torch.no_grad()
    def _update(self):
        if not DIS["soft"]:
            return PixelImage._orig_update(self)
        if self.current_gamma < self.target_gamma:
            self.current_gamma = min(self.current_gamma + self.gamma_step, self.target_gamma)
            if self.hdr_loss is not None:
                self.hdr_loss.gamma = self.current_gamma
        self.pallet.copy_(self.pallet.clamp(0, self.pallet_inertia))
        # value left UNCLAMPED - tanh squash in decode keeps it in range with live gradients
        self.tensor.copy_(self.tensor.clamp(0, float("inf")))
        return self.get_image_tensor()
    PixelImage.update = _update
    print("[ok] gumbel/soft-clamp decode patch installed (gumbel:", DIS["gumbel"], "| soft:", DIS["soft"], ")")
except Exception:
    traceback.print_exc()

# ---- (c) Spectral (Fourier 1/f) + (d) DeepDream-style gradient preconditioning ----
def _spectral_whiten(g, alpha):
    G = torch.fft.rfft2(g.float())
    H = g.shape[-2]; Wf = G.shape[-1]
    fy = torch.fft.fftfreq(H, device=g.device).abs().view(-1, 1)
    fx = torch.fft.rfftfreq(g.shape[-1], device=g.device).abs().view(1, -1)
    f = torch.sqrt(fy * fy + fx * fx); f[0, 0] = 1.0
    mult = 1.0 / (f ** alpha)
    mult = mult / mult.mean()                 # preserve overall gradient energy
    out = torch.fft.irfft2(G * mult, s=g.shape[-2:]).to(g.dtype)
    return out * (g.norm() / (out.norm() + 1e-12))   # norm-preserving: reshape only, no runaway->black

def _deepdream_norm(g):
    return g / (g.abs().mean() + 1e-8)

# ---- depth-guided preconditioning: use the depth model's shapes to shape the gradient ----
def _box(x, r):
    k = 2 * r + 1
    return F.avg_pool2d(F.pad(x[None, None], (r, r, r, r), mode="reflect"), k, stride=1)[0, 0]

def _guided_filter(p, guide, r, eps):
    # He et al. guided filter: smooth p where `guide` (depth) is flat, preserve p across depth edges
    mI, mp = _box(guide, r), _box(p, r)
    varI = _box(guide * guide, r) - mI * mI
    covIp = _box(guide * p, r) - mI * mp
    a = covIp / (varI + eps)
    b = mp - a * mI
    return _box(a, r) * guide + _box(b, r)

def _layer_group(g, guide, n_layers):
    # quantize depth into n bands; every pixel in a band gets that band's mean gradient
    idx = (guide.clamp(0, 1) * (n_layers - 1)).round().long()
    out = torch.zeros_like(g)
    for bnd in range(n_layers):
        m = (idx == bnd)
        if m.any():
            out[m] = g[m].mean()
    return out

# module-global cache populated by zoom_3d's per-frame depth call (see _install_depth_cache)
if not hasattr(builtins, "_DG_CACHE"):
    builtins._DG_CACHE = {"depth": None}

def _resize_guide(d, img):
    import numpy as _np
    d = torch.as_tensor(_np.asarray(d), dtype=torch.float32, device=img.value.device)
    while d.dim() > 2:
        d = d.squeeze(0)
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    H, W = img.value.shape
    return F.interpolate(d[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]

def _depth_guide_for(img):   # fallback only: own depth pass when no zoom_3d depth exists (2D/off)
    from pytti.LossAug.DepthLoss import DepthLoss
    with torch.no_grad():
        d, _ = DepthLoss.get_depth(img.decode_image())
    return _resize_guide(d, img)

def _install_depth_cache():
    # wrap whatever DepthLoss.get_depth currently is (AdaBins or MiDaS) so the depth zoom_3d
    # already computes once per frame is cached + reused by the hook - no extra depth pass
    from pytti.LossAug.DepthLoss import DepthLoss
    cur = DepthLoss.get_depth
    if getattr(cur, "_dg_caching", False):
        return
    def cached(pil_image, *a, **k):
        out = cur(pil_image, *a, **k)
        try:
            builtins._DG_CACHE["depth"] = out[0] if isinstance(out, tuple) else out
        except Exception:
            pass
        return out
    cached._dg_caching = True
    DepthLoss.get_depth = staticmethod(cached)

def _apply_depth_guided(img, g):
    step = getattr(img, "_dg_step", 0); img._dg_step = step + 1
    if step < DIS["dg_warmup"]:
        return g
    cache = builtins._DG_CACHE.get("depth")
    if cache is not None:                       # reuse zoom_3d's per-frame depth
        tok = id(cache)                         # resize only when the frame's depth changes
        if getattr(img, "_dg_tok", None) != tok:
            try:
                img._dg_guide = _resize_guide(cache, img); img._dg_tok = tok
            except Exception:
                traceback.print_exc(); return g
        d = getattr(img, "_dg_guide", None)
    else:                                       # 2D / off mode: own periodic pass
        if getattr(img, "_dg_guide", None) is None or (step % max(1, DIS["dg_refresh"]) == 0):
            try:
                img._dg_guide = _depth_guide_for(img)
            except Exception:
                traceback.print_exc(); return g
        d = getattr(img, "_dg_guide", None)
    if d is None or d.shape != g.shape:
        return g
    if DIS["dg_mode"] == "layers":
        return _layer_group(g, d, DIS["dg_layers"])
    return _guided_filter(g, d, DIS["dg_radius"], DIS["dg_eps"])

try:
    if not hasattr(PixelImage, "_orig_init"):
        PixelImage._orig_init = PixelImage.__init__
    def _init(self, *a, **k):
        PixelImage._orig_init(self, *a, **k)
        def hook(grad):
            g = grad
            try:
                if DIS["spectral"]:
                    g = _spectral_whiten(g, DIS["alpha"])
                if DIS["laplacian"]:
                    g = _deepdream_norm(g)
                if DIS["depth_guided"]:
                    g = _apply_depth_guided(self, g)
            except Exception:
                return grad
            return g
        self.value.register_hook(hook)
    PixelImage.__init__ = _init
    if DIS["depth_guided"]:
        _install_depth_cache()
    print("[ok] gradient-preconditioning hook installed",
          "(spectral:", DIS["spectral"], "| laplacian:", DIS["laplacian"],
          "| depth_guided:", DIS["depth_guided"], "->", DIS["dg_mode"], ")")
except Exception:
    traceback.print_exc()

# ---- (e) curated palette from an image (helper) ----
#@markdown To use a curated palette from an image, set `target_palette` in 2.1 to its path
#@markdown (p5 already extracts + locks a palette from it). `make_palette_image(path, n)`
#@markdown below builds a clean N-colour swatch you can save and reuse.
try:
    from PIL import Image
    def make_palette_image(path, n_colors=16):
        img = Image.open(path).convert("RGB")
        q = img.convert("P", palette=Image.ADAPTIVE, colors=n_colors).convert("RGB")
        return q
    print("[ok] make_palette_image(path, n_colors) available")
except Exception:
    traceback.print_exc()


In [ ]:
#@title 2.17.1  Init-image colour-fidelity fix (encode no longer darkens)
#@markdown Bugfix: Limited-Palette `encode_image` wrote the palette in ~[0,1] but `decode` divides by
#@markdown `pallet_inertia` (=2), and it bucketed brightness by `value/value.max()` while decode
#@markdown indexes by TRUE luma -> the init image came out ~2x darker / mis-toned. This patches encode
#@markdown so the encoded image matches the original. (Also fixed in source PixelImage.py.) Run before 2.3.
import torch, traceback
import torch.nn.functional as F
from torchvision.transforms import functional as TF
from PIL import Image
from pytti import DEVICE
from pytti.Image.PixelImage import PixelImage

fix_encode_color = True  #@param{type:"boolean"}
#@markdown start at full gamma so frame 0 already has the intended saturation (no washed->saturated ramp over the first ~120 frames)
start_at_full_gamma = True  #@param{type:"boolean"}
#@markdown counter the slight palette wash-out (boosts chroma, keeps brightness). 1.0=off, ~1.2 nice
init_saturation = 1.0  #@param{type:"number"}

try:
    if not hasattr(PixelImage, "_orig_encode_image_color"):
        PixelImage._orig_encode_image_color = PixelImage.encode_image

    def _encode_image_fixed(self, pil_image, smart_encode=True, device=DEVICE):
        if not fix_encode_color:
            return PixelImage._orig_encode_image_color(self, pil_image, smart_encode=smart_encode, device=device)
        width, height = self.image_shape
        scale = self.scale
        color_ref = pil_image.resize((width // scale, height // scale), Image.LANCZOS)
        color_ref = TF.to_tensor(color_ref).to(device)
        with torch.no_grad():
            magic_color = torch.tensor([0.299, 0.587, 0.114], device=device).view(3, 1, 1)
            value_ref = (color_ref * magic_color).sum(dim=0)
            self.value.copy_(value_ref)
            if start_at_full_gamma:
                self.current_gamma = self.target_gamma   # frame 0 already at intended gamma/saturation

            pallet_size = self.pallet_size
            n_pallets = self.n_pallets
            # quantize by TRUE luma (decode indexes by true luma, not value/max); no div-by-zero
            value_quantized = (self.value.clamp(0, 1) * (pallet_size - 1)).long().clamp(0, pallet_size - 1)

            epsilon = 1e-6
            normalized_color = color_ref / (self.value.unsqueeze(0) + epsilon)
            normalized_color = normalized_color.permute(1, 2, 0).contiguous().view(-1, 3)
            value_quantized_flat = value_quantized.view(-1)

            # prefill grayscale brightness ramp so every level is populated & brightness-monotonic
            # (empty levels -> sort_pallet scrambles colours, esp. reds)
            _ramp = (torch.arange(pallet_size, device=device, dtype=self.pallet.dtype)
                     / max(1, pallet_size - 1) * self.pallet_inertia).view(pallet_size, 1, 1)
            new_pallet = _ramp.repeat(1, n_pallets, 3).contiguous()
            new_tensor = torch.zeros_like(self.tensor)

            from sklearn.cluster import KMeans
            w = self.value.shape[1]
            for i in range(pallet_size):
                mask = (value_quantized_flat == i)
                if mask.sum() == 0:
                    continue
                colors = normalized_color[mask]
                n_clusters = min(n_pallets, colors.shape[0])
                if n_clusters == 0:
                    continue
                kmeans = KMeans(n_clusters=n_clusters, n_init=1, max_iter=10, random_state=0)
                labels = kmeans.fit_predict(colors.cpu().numpy())
                cluster_centers = torch.tensor(kmeans.cluster_centers_, device=device)
                if init_saturation != 1.0:
                    cluster_centers = 1.0 + init_saturation * (cluster_centers - 1.0)  # boost chroma, keep luma/brightness
                brightness_value = (i / (pallet_size - 1))
                # *pallet_inertia so decode's (pallet / pallet_inertia) reproduces the original colour
                new_pallet[i, :n_clusters, :] = cluster_centers * brightness_value * self.pallet_inertia
                if n_clusters < n_pallets:
                    new_pallet[i, n_clusters:, :] = cluster_centers[-1] * brightness_value * self.pallet_inertia
                indices = torch.nonzero(mask).squeeze(-1)
                labels = torch.tensor(labels, device=device).long()
                new_tensor[labels, (indices // w).long(), (indices % w).long()] = 1
            new_tensor = new_tensor.clamp(0, 1)
            self.pallet.copy_(new_pallet)
            self.tensor.copy_(new_tensor)

    PixelImage.encode_image = _encode_image_fixed
    print("[ok] init-image colour-fidelity encode fix installed (enabled:", fix_encode_color, ")")
except Exception:
    traceback.print_exc()


In [ ]:
#@title 2.18  NCA - Step 1: Setup
#@markdown Trains a cellular-automata rule to GROW an image, then applies it as a frozen local-detail
#@markdown overlay on Limited Palette. Target image + resolution come from 2.1 (init_image, width, height).
import torch, builtins
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from PIL import Image
from pytti import DEVICE
from pytti.Image import DifferentiableImage
from pytti.Image.GNCAImage import GNCAImage, CAModel
from pytti.Image.PixelImage import PixelImage

nca_channels        = 16   #@param{type:"number"}
nca_steps           = 48   #@param{type:"number"}
nca_fire_rate       = 1.0  #@param{type:"number"}
nca_init_scale      = 0.0  #@param{type:"number"}
nca_step_size       = 0.5  #@param{type:"number"}
nca_overflow_weight = 1.0  #@param{type:"number"}
nca_learning_rate   = 1e-3 #@param{type:"number"}

NCA = dict(channels=int(nca_channels), steps=int(nca_steps), fire=float(nca_fire_rate),
           step=float(nca_step_size), overflow=float(nca_overflow_weight),
           lr=float(nca_learning_rate), init_scale=float(nca_init_scale), mix=0.5, overlay=False)
builtins.NCA = NCA
if not hasattr(builtins, "PAL_NCA_CA"): builtins.PAL_NCA_CA = None

def _to_rgb(state):
    rgb = torch.sigmoid(state[:, :3])
    rgb = torch.nan_to_num(rgb, nan=0.5, posinf=1.0, neginf=0.0) * (1.0 - 4e-3)
    tilt = torch.tensor([1e-3, 2e-3, 3e-3], device=rgb.device, dtype=rgb.dtype).view(1, 3, 1, 1)
    return rgb + tilt                                   # in (0,1), channels differ -> hue-safe for kornia

def _seed(channels, h, w, device=DEVICE):
    return torch.randn(1, channels, h, w, device=device) * NCA["init_scale"]

def _logit(p, eps=1e-3):                              # feed an image into the state so sigmoid(seed)==image
    p = p.clamp(eps, 1.0 - eps)
    return torch.log(p / (1.0 - p))

def _run(ca, x, training):
    if training:
        if not x.requires_grad: x = x.requires_grad_(True)
        for _ in range(NCA["steps"]):
            x = checkpoint(ca, x, use_reentrant=False)  # gradient checkpointing -> big images fit in VRAM
    else:
        for _ in range(NCA["steps"]):
            x = ca(x)
    return x

def _perceive(self, x, angle=0.0):                      # correct depthwise sobel perception
    base = torch.stack([self.identity, self.dx, self.dy], dim=0)
    cn = self.channel_n
    w = base.repeat(cn, 1, 1).reshape(3 * cn, 1, 3, 3).to(x.dtype)
    return F.conv2d(x, w, padding=1, groups=cn)
CAModel.perceive = _perceive
def _ca_forward(self, x, fire_rate=None, angle=0.0, step_size=1.0):
    dx = self.dmodel(self.perceive(x)) * NCA["step"]
    if self.fire_rate >= 1.0:
        x = x + dx                                   # deterministic update (no flicker)
    else:
        m = (torch.rand_like(x[:, :1]) <= self.fire_rate).float()
        x = x + dx * m
    return torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0).clamp(-10.0, 10.0)
CAModel.forward = _ca_forward

def _init(self, width, height, scale=1, device=DEVICE, **kw):
    DifferentiableImage.__init__(self, width, height)
    self.channel_n = int(NCA["channels"]); self.scale = scale
    self.ca_model = CAModel(channel_n=self.channel_n, fire_rate=float(NCA["fire"])).to(device)
    self.register_buffer("state", _seed(self.channel_n, height, width, device))
    self.register_buffer("target", torch.zeros(1, 3, height, width, device=device))
    self._last = None
    self.output_axes = ('n', 's', 'y', 'x'); self.lr = NCA["lr"]
    for p in self.ca_model.parameters():
        p.register_hook(lambda g: g / (g.norm() + 1e-8))   # NCA gradient normalization
GNCAImage.__init__ = _init
def _set_target(self, pil):
    from torchvision.transforms import functional as TF
    w, h = self.image_shape
    pil = pil.convert("RGB").resize((w, h), Image.LANCZOS)
    self.target.copy_(TF.to_tensor(pil).unsqueeze(0).to(self.target.device))
GNCAImage.set_target = _set_target
def _decode_training_tensor(self):
    x = _run(self.ca_model, self.state.detach(), True)
    self._overflow = (x.abs() - 1.0).clamp(min=0.0).mean()
    self._last = x.detach()
    return _to_rgb(x)
GNCAImage.decode_training_tensor = _decode_training_tensor
@torch.no_grad()
def _decode_tensor(self):
    return _to_rgb(self.state)                            # render the grown pool
GNCAImage.decode_tensor = _decode_tensor
@torch.no_grad()
def _update(self):
    if self._last is not None:
        self.state.copy_(self._last.clamp(-3, 3))         # persist the grown state
GNCAImage.update = _update
GNCAImage.decode_image = DifferentiableImage.decode_image

def _overlay(pal_rgb):
    ca = builtins.PAL_NCA_CA
    if ca is None: return pal_rgb
    extra = _seed(ca.channel_n, pal_rgb.shape[-2], pal_rgb.shape[-1], pal_rgb.device)[:, 3:]
    x = torch.cat([_logit(pal_rgb), extra], dim=1)                # seed the frozen rule with the palette image
    x = _run(ca, x, torch.is_grad_enabled())
    return (1.0 - NCA["mix"]) * pal_rgb + NCA["mix"] * _to_rgb(x)
if not getattr(PixelImage, "_overlay_installed", False):
    PixelImage._pre_overlay_decode = PixelImage.decode_tensor
    def _pix_decode(self):
        rgb = PixelImage._pre_overlay_decode(self)
        return _overlay(rgb) if (NCA["overlay"] and builtins.PAL_NCA_CA is not None) else rgb
    PixelImage.decode_tensor = _pix_decode
    PixelImage._overlay_installed = True
print("[ok] NCA setup ready. (target + resolution come from 2.1)")


In [ ]:
#@title 2.18.1  NCA - Step 2: Train the rule (local detail operator)
#@markdown Trains the rule as a LOCAL DETAIL operator: seed = a blurred/degraded crop, target = the
#@markdown sharp crop. It learns "given low-detail local content, add detail" - which is exactly what
#@markdown it does when run over the Limited Palette. Converges (unlike grow-from-noise). Crops = fast.
import torch, os, copy, builtins, random
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
from IPython.display import display
from PIL import Image
from pytti import fetch

if "_logit" not in dir():
    def _logit(p, eps=1e-3):
        p = p.clamp(eps, 1.0 - eps); return torch.log(p / (1.0 - p))

train_steps    = 6000 #@param{type:"number"}
train_crop     = 128  #@param{type:"number"}
degrade_factor = 4    #@param{type:"number"}
use_amp        = True #@param{type:"boolean"}
preview_every  = 100  #@param{type:"number"}
save_path      = "nca_models/my_nca.pt" #@param{type:"string"}

assert str(params.init_image).strip(), "Set init_image in 2.1 - the NCA learns to detail that image."
W, H = int(params.width), int(params.height)
NCA_TRAINED = GNCAImage(W, H)
NCA_TRAINED.set_target(Image.open(fetch(params.init_image)).convert("RGB"))
ca = NCA_TRAINED.ca_model
opt = optim.Adam(ca.parameters(), lr=NCA["lr"])
os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
crop = int(train_crop); crop = crop if (0 < crop < min(W, H)) else 0
amp  = bool(use_amp) and torch.cuda.is_available()
deg_f = max(1, int(degrade_factor))

def _degrade(t):                                        # lose high-freq detail (mimics the palette input)
    if deg_f <= 1: return t
    h, w = t.shape[-2], t.shape[-1]
    s = F.interpolate(t, size=(max(1, h // deg_f), max(1, w // deg_f)), mode="bilinear", align_corners=False)
    return F.interpolate(s, size=(h, w), mode="bilinear", align_corners=False)

def _grow_from(content, training):
    extra = _seed(NCA_TRAINED.channel_n, content.shape[-2], content.shape[-1], content.device)[:, 3:]
    x = torch.cat([_logit(_degrade(content)), extra], dim=1)  # seed first 3 channels = degraded content
    if training and not crop:
        if not x.requires_grad: x = x.requires_grad_(True)
        for _ in range(NCA["steps"]): x = checkpoint(ca, x, use_reentrant=False)
    else:
        for _ in range(NCA["steps"]): x = ca(x)
    return x

def _save():
    builtins.NCA_RULE = copy.deepcopy(ca.state_dict())
    torch.save({"state_dict": ca.state_dict(), "channels": NCA_TRAINED.channel_n}, save_path)

@torch.no_grad()
def _preview():
    x = _grow_from(NCA_TRAINED.target, False)
    a = _to_rgb(x)[0].permute(1, 2, 0).clamp(0, 1).mul(255).byte().cpu().numpy()
    return Image.fromarray(a)

mode = f"{crop}x{crop} crops" if crop else f"full {W}x{H} (checkpointed)"
print(f"training detail operator on {params.init_image} | {mode} | degrade x{deg_f} | amp={amp} | {train_steps} steps")
handle = display(_preview(), display_id=True)
pbar = tqdm(range(int(train_steps)), desc="NCA training")
ema = None
try:
    for step in pbar:
        opt.zero_grad()
        with torch.cuda.amp.autocast(enabled=amp):
            if crop:
                cy = random.randint(0, H - crop); cx = random.randint(0, W - crop)
                content = NCA_TRAINED.target[..., cy:cy + crop, cx:cx + crop]
            else:
                content = NCA_TRAINED.target
            x = _grow_from(content, True)
            rgb = _to_rgb(x); ov = (x[:, 4:].abs() - 1.0).clamp(min=0.0).mean()  # hidden channels only
            loss = F.mse_loss(rgb.float(), content) + NCA["overflow"] * ov.float()
        loss.backward(); opt.step()
        _l = loss.item(); ema = _l if ema is None else 0.98 * ema + 0.02 * _l
        pbar.set_postfix(loss=f"{_l:.4f}", ema=f"{ema:.4f}")
        if step % max(1, int(preview_every)) == 0:
            _save(); handle.update(_preview()); pbar.write(f"step {step}: loss {_l:.4f} | ema {ema:.4f} (saved)")
except KeyboardInterrupt:
    pbar.write("interrupted -> saving")
_save()
print("done. detail rule in memory (NCA_RULE) + saved ->", save_path)
display(_preview())


In [ ]:
#@title 2.19  NCA - Step 3: Watch the detailer run (playback)
#@markdown Seeds from the image, re-injects it every `play_reinject` steps (keeps it near the image and
#@markdown re-detailing -> alive), and swaps random neighboring pixels each step for organic motion.
import torch, os, imageio, builtins
from PIL import Image
from IPython.display import display, Video

if "_logit" not in dir():
    def _logit(p, eps=1e-3):
        p = p.clamp(eps, 1.0 - eps); return torch.log(p / (1.0 - p))

play_steps     = 400  #@param{type:"number"}
play_reinject  = 0    #@param{type:"number"}
play_swap_rate = 0.02 #@param{type:"number"}
play_fps       = 24   #@param{type:"number"}
assert 'NCA_TRAINED' in dir(), "Train first (Step 2)."
reinject = int(play_reinject) if int(play_reinject) > 0 else NCA["steps"]
out = "images_out/nca_play"; os.makedirs(out, exist_ok=True)
frames = []
with torch.no_grad():
    h, w = NCA_TRAINED.target.shape[-2:]
    seed3 = _logit(NCA_TRAINED.target)
    extra = _seed(NCA_TRAINED.channel_n, h, w)[:, 3:]
    x = torch.cat([seed3, extra], dim=1)                        # frame 0 = the image (logit seed)
    for s in range(int(play_steps)):
        if s > 0 and s % reinject == 0:
            x[:, :3] = seed3                                    # re-anchor to the image; keeps re-detailing
        x = NCA_TRAINED.ca_model(x)
        if play_swap_rate > 0:
            # swap random pixels with a neighbor for organic motion (no energy injection)
            mask = torch.rand(1, 1, h, w, device=x.device) < float(play_swap_rate)
            direction = torch.randint(0, 4, (1,), device=x.device).item()
            if direction == 0:   shifted = torch.roll(x, 1, dims=-1)   # right
            elif direction == 1: shifted = torch.roll(x, -1, dims=-1)  # left
            elif direction == 2: shifted = torch.roll(x, 1, dims=-2)   # down
            else:                shifted = torch.roll(x, -1, dims=-2)  # up
            x = torch.where(mask, shifted, x)
        frames.append(_to_rgb(x)[0].permute(1, 2, 0).clamp(0, 1).mul(255).byte().cpu().numpy())
vid = f"{out}/play.mp4"; imageio.mimsave(vid, frames, fps=int(play_fps))
print("saved", len(frames), "frames ->", vid)
try: display(Video(vid, embed=True, width=512))
except Exception: display(Image.fromarray(frames[-1]))


In [ ]:
#@title 2.18.2  NCA - Step 4: Apply as local detail on Limited Palette
import builtins, torch
from pytti import DEVICE
from pytti.Image.GNCAImage import CAModel

apply_nca_detail = True #@param{type:"boolean"}
detail_mix       = 0.25 #@param{type:"number"}

if apply_nca_detail:
    assert getattr(builtins, "NCA_RULE", None) is not None, "Train the rule first (Step 2)."
    ca = CAModel(channel_n=int(NCA["channels"]), fire_rate=float(NCA["fire"])).to(DEVICE)
    ca.load_state_dict(builtins.NCA_RULE); ca.eval()
    for p in ca.parameters(): p.requires_grad_(False)
    builtins.PAL_NCA_CA = ca
    NCA["overlay"] = True; NCA["mix"] = float(detail_mix)
    print("Overlay ON (mix", NCA["mix"], "). Keep image_model='Limited Palette' in 2.1, then run 2.3.")
else:
    NCA["overlay"] = False; builtins.PAL_NCA_CA = None
    print("Overlay OFF - plain Limited Palette.")


In [ ]:
#@title 2.20  Second motion layer: diffuse / shuffle / swirl / drift / noise
#@markdown A **second animation layer** that runs *on top of* whatever 2.1 `animation_mode` is doing
#@markdown (2D, 3D, **or `off`**). After each frame's normal motion it applies extra, non-affine
#@markdown transforms to the image - colour diffusion, NCA-style pixel shuffle, a radial swirl, a
#@markdown directional drift, and noise injection. All default OFF; nothing is touched unless you
#@markdown flip `enable_secondary_motion`. Run AFTER **2.1**, BEFORE **2.3**. Re-run to change a knob.

import torch, math, traceback, builtins
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import functional as TF
from pytti import DEVICE
from pytti.ImageGuide import DirectImageGuide
from pytti.Notebook import tqdm
from pytti.Transforms import apply_grid, PADDING_MODES

#@markdown #### Master switch
enable_secondary_motion = False  #@param{type:"boolean"}
#@markdown apply the layer every N frames (1 = every frame)
motion_every_frames     = 1      #@param{type:"number"}
#@markdown #### Diffusion (colours bleed into one another, heat-equation style)
diffuse_amount          = 0.0    #@param{type:"number"}
diffuse_radius          = 2      #@param{type:"number"}
#@markdown #### Pixel shuffle (NCA `fire_rate` homage - jitter a fraction of pixels each frame)
shuffle_amount          = 0.0    #@param{type:"number"}
shuffle_radius          = 1      #@param{type:"number"}
#@markdown #### Swirl (radius-dependent rotation about a point - 2D can't do this)
swirl_strength          = 0.0    #@param{type:"number"}
swirl_sigma             = 0.5    #@param{type:"number"}
swirl_center_x          = 0.0    #@param{type:"number"}
swirl_center_y          = 0.0    #@param{type:"number"}
#@markdown #### Drift (smooth directional push, in pixels/frame, at an angle in degrees)
drift_amount            = 0.0    #@param{type:"number"}
drift_direction         = 0.0    #@param{type:"number"}
#@markdown #### Noise injection (relative to local contrast)
noise_amount            = 0.0    #@param{type:"number"}
#@markdown #### Sampling for the warp ops
border_mode_secondary   = "mirror"   #@param ["mirror","smear","black","wrap"]
sampling_mode_secondary = "bilinear" #@param ["bilinear","nearest","bicubic"]

MOT = dict(
    enabled=enable_secondary_motion, every=max(1, int(motion_every_frames)),
    diffuse=float(diffuse_amount), diffuse_radius=int(diffuse_radius),
    shuffle=float(shuffle_amount), shuffle_radius=int(shuffle_radius),
    swirl=float(swirl_strength), swirl_sigma=float(swirl_sigma),
    swirl_cx=float(swirl_center_x), swirl_cy=float(swirl_center_y),
    drift=float(drift_amount), drift_dir=float(drift_direction),
    noise=float(noise_amount),
    border=border_mode_secondary, sampling=sampling_mode_secondary,
)
builtins.MOT = MOT

# --- get/set the image tensor exactly like Transforms.zoom_2d (with PIL fallback) ----------
def _get_tensor(img):
    try:
        return img.get_image_tensor().unsqueeze(0), False
    except NotImplementedError:
        return TF.to_tensor(img.decode_image()).unsqueeze(0).to(DEVICE), True

def _set_tensor(img, tensor, fallback):
    t = tensor.squeeze(0)
    if not fallback:
        img.set_image_tensor(t)
    else:
        arr = t.movedim(0, -1).mul(255).clamp(0, 255).byte().cpu().numpy()
        img.encode_image(Image.fromarray(arr))

def _identity_grid(shape):
    eye = torch.eye(3, device=DEVICE)[:2].unsqueeze(0)
    return F.affine_grid(eye, shape, align_corners=True)   # (1, H, W, 2) -> [...,0]=x, [...,1]=y in [-1,1]

# --- the layer: build one displacement field (drift+swirl+shuffle), sample once, then diffuse+noise
@torch.no_grad()
def _apply_secondary_motion(img):
    t, fallback = _get_tensor(img)
    H, W = t.shape[-2:]
    g = _identity_grid(t.shape)                            # base sampling coords
    gx, gy = g[..., 0], g[..., 1]
    off = torch.zeros_like(g)

    # swirl: rotate sampling coords about (cx,cy) by an angle that decays with radius
    if MOT["swirl"] != 0.0:
        dx = gx - MOT["swirl_cx"]; dy = gy - MOT["swirl_cy"]
        r2 = dx * dx + dy * dy
        ang = MOT["swirl"] * torch.exp(-r2 / (2 * MOT["swirl_sigma"] ** 2 + 1e-8))
        ca, sa = torch.cos(ang), torch.sin(ang)
        nx = MOT["swirl_cx"] + dx * ca - dy * sa
        ny = MOT["swirl_cy"] + dx * sa + dy * ca
        off[..., 0] += nx - gx; off[..., 1] += ny - gy

    # drift: constant directional push (pixels/frame); sample from the opposite side -> content moves
    if MOT["drift"] != 0.0:
        th = math.radians(MOT["drift_dir"])
        off[..., 0] += -2.0 * MOT["drift"] * math.cos(th) / W
        off[..., 1] += -2.0 * MOT["drift"] * math.sin(th) / H

    # pixel shuffle: jitter a random fraction of pixels by +/- shuffle_radius px (NCA fire_rate idea)
    if MOT["shuffle"] > 0.0:
        r = max(1, MOT["shuffle_radius"])
        mask = (torch.rand(1, H, W, device=DEVICE) <= MOT["shuffle"]).float()
        jx = torch.randint(-r, r + 1, (1, H, W), device=DEVICE).float() * mask
        jy = torch.randint(-r, r + 1, (1, H, W), device=DEVICE).float() * mask
        off[..., 0] += jx * 2.0 / W; off[..., 1] += jy * 2.0 / H

    if off.abs().sum() > 0:
        t = apply_grid(t, g + off, MOT["border"], MOT["sampling"])

    # diffusion: blend toward a blurred copy so neighbouring colours bleed
    if MOT["diffuse"] > 0.0:
        rad = max(1, MOT["diffuse_radius"]); k = 2 * rad + 1
        blurred = TF.gaussian_blur(t, [k, k], [rad / 2.0, rad / 2.0])
        a = min(1.0, MOT["diffuse"])
        t = (1 - a) * t + a * blurred

    # noise: scaled by the tensor's own std so it reads the same across image models
    if MOT["noise"] > 0.0:
        t = t + torch.randn_like(t) * (MOT["noise"] * (t.std() + 1e-6))

    _set_tensor(img, t, fallback)

# --- frame boundary detection mirrors the update() logic in 2.3 (works for animation_mode 'off') --
def _is_frame_boundary(i):
    p = globals().get("params", getattr(builtins, "params", None))
    if p is None:
        return False
    pre = int(getattr(p, "pre_animation_steps", 0))
    spf = max(1, int(getattr(p, "steps_per_frame", 1)))
    if i < pre or (i - pre) % spf != 0:
        return False
    frame = (i - pre) // spf
    return (frame % MOT["every"]) == 0

# --- patch run_steps so the extra layer fires once per frame, AFTER the normal motion in update() --
try:
    if not hasattr(DirectImageGuide, "_orig_run_steps_mot"):
        DirectImageGuide._orig_run_steps_mot = DirectImageGuide.run_steps

    def _run_steps_mot(self, n_steps, prompts, interp_prompts, loss_augs,
                       stop=-math.inf, interp_steps=0, i_offset=0, skipped_steps=0):
        for i in tqdm(range(n_steps)):
            gi = i + i_offset
            self.update(gi, i + skipped_steps)            # legacy per-frame 2D/3D motion happens here
            if MOT["enabled"]:
                try:
                    if _is_frame_boundary(gi):
                        _apply_secondary_motion(self.image_rep)
                except Exception:
                    traceback.print_exc()
            losses = self.train(i + skipped_steps, prompts, interp_prompts, loss_augs,
                                interp_steps=interp_steps)
            if losses["TOTAL"] <= stop:
                break
        return i + 1

    DirectImageGuide.run_steps = _run_steps_mot
    print("[ok] second motion layer installed (enabled:", MOT["enabled"],
          "| every:", MOT["every"], "frame(s))")
    if MOT["enabled"]:
        active = {k: v for k, v in MOT.items()
                  if k in ("diffuse", "shuffle", "swirl", "drift", "noise") and v}
        print("     active ops:", active or "(none - set an *_amount/strength > 0)")
        print("     note: with stabilization weights on, the layer competes with them;",
              "set animation_mode='off' or lower those weights for the strongest effect.")
except Exception:
    traceback.print_exc()


In [ ]:
#@title 2.21  Shape-spectral preconditioning  (+ scale band + guide coupling)
#@markdown Whitens the gradient in Fourier space so structure pops -- with THREE controls now:
#@markdown 1) a **shape** envelope (circle/polygon/spiral/fire) instead of plain 1/f;
#@markdown 2) a **scale band** that picks the feature SIZE (in px) that coheres -- e.g. only
#@markdown    sub-cutout detail, or only big global blobs; and
#@markdown 3) a **guide** (reaction-diffusion sim / a video / an image) that makes structure form
#@markdown    around its bright regions ("attract") or only develop there ("mask").
#@markdown Independent of 2.17. Default OFF. Run AFTER 2.1, BEFORE 2.3. Limited Palette (PixelImage).
#@markdown NOTE: "px" here = value-map pixels = canvas / pixel_size  (== output px when pixel_size=1).

import torch, math, traceback, builtins
import torch.nn.functional as F
from torchvision.transforms import functional as TF
from pytti.Image.PixelImage import PixelImage

shape_spectral   = False   #@param{type:"boolean"}
shape_type       = "circle"  #@param ["circle", "polygon", "spiral", "fire"]
shape_alpha      = 1.0     #@param{type:"number"}
#@markdown how much of the envelope is the shape vs. classic radial 1/f (1.0 = pure shape)
shape_mix        = 0.85    #@param{type:"number"}
shape_radius     = 0.5     #@param{type:"number"}
shape_center_x   = 0.0     #@param{type:"number"}
shape_center_y   = 0.0     #@param{type:"number"}
polygon_sides    = 3       #@param{type:"number"}
shape_rotation   = 0.0     #@param{type:"number"}
spiral_arms      = 2       #@param{type:"number"}
spiral_tightness = 6.0     #@param{type:"number"}
shape_speed      = 0.15    #@param{type:"number"}
shape_animate    = True    #@param{type:"boolean"}
shape_softness   = 0.06    #@param{type:"number"}

#@markdown #### Scale band  --  pick the feature SIZE (px) that gets coherence
scale_band       = False   #@param{type:"boolean"}
#@markdown smallest feature kept (px). e.g. 8
scale_min_px     = 8       #@param{type:"number"}
#@markdown largest feature kept (px). set ~= a cutout size to suppress GLOBAL shape and keep
#@markdown coherence "inside the cutout". make it big for large global blobs.
scale_max_px     = 128     #@param{type:"number"}
#@markdown band-edge softness in log-frequency (smaller = sharper band)
scale_soft       = 0.5     #@param{type:"number"}
#@markdown #### Detail floor -- stop the envelope SUPPRESSING high freqs (counteracts zoom
#@markdown "stretching" so fine detail keeps regenerating). 0 = pure spectral, ~0.6-1.0 = keep detail.
detail_floor     = 0.0     #@param{type:"number"}

#@markdown #### Guide  --  make structure form around a field (bright -> lighter)
guide_enabled    = False   #@param{type:"boolean"}
guide_source     = "reaction_diffusion"  #@param ["reaction_diffusion", "video", "image"]
guide_path       = ""      #@param{type:"string"}
guide_mode       = "attract"  #@param ["attract", "mask"]
#@markdown attract: pull luma toward the guide (forms shapes around it). mask: gate where
#@markdown structure develops (0..1). strength tunes both.
guide_strength   = 0.15    #@param{type:"number"}
guide_invert     = False   #@param{type:"boolean"}
guide_blur_px    = 0       #@param{type:"number"}
#@markdown reaction-diffusion (Gray-Scott): pattern controls + sim iters/step + sim grid cap
rd_feed          = 0.055   #@param{type:"number"}
rd_kill          = 0.062   #@param{type:"number"}
rd_iters         = 8       #@param{type:"number"}
rd_res           = 192     #@param{type:"number"}
#@markdown video: advance one source frame every N optimizer steps
guide_video_stride = 1     #@param{type:"number"}

SHP = dict(enabled=shape_spectral, shape=shape_type, alpha=float(shape_alpha),
           mix=float(shape_mix), radius=float(shape_radius),
           cx=float(shape_center_x), cy=float(shape_center_y),
           sides=max(3, int(polygon_sides)), rot=float(shape_rotation),
           arms=max(1, int(spiral_arms)), tight=float(spiral_tightness),
           speed=float(shape_speed), animate=bool(shape_animate),
           soft=max(1e-3, float(shape_softness)),
           band=bool(scale_band), smin_px=float(scale_min_px), smax_px=float(scale_max_px),
           ssoft=max(1e-3, float(scale_soft)), detail_floor=max(0.0, float(detail_floor)),
           guide=bool(guide_enabled), guide_src=guide_source, guide_path=guide_path,
           guide_mode=guide_mode, gstr=float(guide_strength), ginv=bool(guide_invert),
           gblur=int(guide_blur_px), rd_feed=float(rd_feed), rd_kill=float(rd_kill),
           rd_iters=max(1, int(rd_iters)), rd_res=max(16, int(rd_res)),
           vid_stride=max(1, int(guide_video_stride)))
builtins.SHP = SHP

# ---- shape fields in [0,1] over a normalized grid (x left->right, y top->bottom) ----------------
def _coords(H, W, device):
    y = torch.linspace(1, -1, H, device=device).view(-1, 1).expand(H, W)
    x = torch.linspace(-1, 1, W, device=device).view(1, -1).expand(H, W)
    return x, y

def _circle(x, y):
    d = torch.sqrt((x - SHP["cx"]) ** 2 + (y - SHP["cy"]) ** 2)
    return torch.sigmoid((SHP["radius"] - d) / SHP["soft"])

def _polygon(x, y, phase):
    dx = x - SHP["cx"]; dy = y - SHP["cy"]
    n = SHP["sides"]; rot = math.radians(SHP["rot"]) + phase
    sdf = torch.full_like(dx, -1e3)
    for k in range(n):
        a = rot + 2 * math.pi * k / n
        sdf = torch.maximum(sdf, dx * math.cos(a) + dy * math.sin(a))
    return torch.sigmoid((SHP["radius"] - sdf) / SHP["soft"])

def _spiral(x, y, phase):
    dx = x - SHP["cx"]; dy = y - SHP["cy"]
    r = torch.sqrt(dx * dx + dy * dy) + 1e-6
    th = torch.atan2(dy, dx)
    band = 0.5 + 0.5 * torch.cos(SHP["arms"] * th + SHP["tight"] * r - phase)
    window = torch.exp(-(r / (SHP["radius"] + 1e-6)) ** 2)
    return band * window

def _fire(x, y, phase):
    base_y = SHP["cy"] - 0.7
    length = SHP["radius"] * 1.8 * (1.0 + 0.12 * math.sin(phase * 1.7))
    ha = (y - base_y)
    sway = 0.08 * torch.sin(phase + (ha / (length + 1e-6)) * 4.0)
    width = (SHP["radius"] * 0.45) * torch.clamp(1.0 - ha / (length + 1e-6), min=0.0)
    body = torch.exp(-((x - SHP["cx"] - sway) / (width + 1e-3)) ** 2)
    body = body * ((ha > 0) & (ha < length)).float()
    body = body * torch.clamp(ha / (length + 1e-6), 0.0, 1.0)
    return body

_ANIMATED = {"spiral", "fire"}

def _shape_field(H, W, step, device):
    x, y = _coords(H, W, device)
    phase = (step * SHP["speed"]) if (SHP["animate"] and SHP["shape"] in _ANIMATED) else 0.0
    if SHP["shape"] == "polygon":   S = _polygon(x, y, phase)
    elif SHP["shape"] == "spiral":  S = _spiral(x, y, phase)
    elif SHP["shape"] == "fire":    S = _fire(x, y, phase)
    else:                           S = _circle(x, y)
    S = S - S.amin()
    return S / (S.amax() + 1e-8)

# ---- scale-band window: keep feature sizes in [smin_px, smax_px] (size L px <-> freq 1/L) --------
def _band_window(f):
    f_lo = 1.0 / max(1e-6, SHP["smax_px"])      # large features -> low freq edge
    f_hi = 1.0 / max(1e-6, SHP["smin_px"])      # small features -> high freq edge
    lf = torch.log(f.clamp_min(1e-9))
    lo, hi = math.log(max(1e-12, f_lo)), math.log(max(1e-12, f_hi))
    s = SHP["ssoft"]
    w = torch.sigmoid((lf - lo) / s) * torch.sigmoid((hi - lf) / s)   # soft top-hat in log-freq
    w = w.clone(); w[0, 0] = 1.0                                      # keep DC (overall brightness)
    return w

# ---- guide fields (luminance in [0,1] at the value-map resolution) ------------------------------
def _to_luma(t):
    if t.dim() == 3:
        y = 0.2126 * t[0] + 0.7152 * t[1] + 0.0722 * t[2]
    else:
        y = t
    y = y - y.amin()
    return y / (y.amax() + 1e-8)

def _laplace(Z):
    k = torch.tensor([[0.05, 0.2, 0.05], [0.2, -1.0, 0.2], [0.05, 0.2, 0.05]],
                     device=Z.device, dtype=Z.dtype).view(1, 1, 3, 3)
    return F.conv2d(F.pad(Z.view(1, 1, *Z.shape), (1, 1, 1, 1), mode="circular"), k)[0, 0]

def _rd_grid(H, W):
    m = max(H, W); s = min(1.0, SHP["rd_res"] / m)
    return max(16, int(round(H * s))), max(16, int(round(W * s)))

def _rd_init(self, H, W, device):
    rh, rw = _rd_grid(H, W)
    U = torch.ones(rh, rw, device=device)
    V = torch.zeros(rh, rw, device=device)
    n = 24; r = max(2, min(rh, rw) // 24)
    ys = torch.randint(0, rh, (n,)).tolist(); xs = torch.randint(0, rw, (n,)).tolist()
    for yy, xx in zip(ys, xs):
        V[max(0, yy - r):yy + r, max(0, xx - r):xx + r] = 1.0
    self._rd_U, self._rd_V = U, V

def _guide_video(self, H, W, step, device):
    if getattr(self, "_vid", None) is None:
        import imageio
        self._vid = imageio.get_reader(SHP["guide_path"])
        try:    self._vid_n = self._vid.count_frames()
        except Exception: self._vid_n = None
        self._vid_cache = (-1, None)
    idx = step // SHP["vid_stride"]
    if self._vid_n:
        idx = idx % self._vid_n
    if self._vid_cache[0] == idx and self._vid_cache[1] is not None:
        return self._vid_cache[1]
    frame = self._vid.get_data(int(idx))                          # HxWx3 uint8
    t = torch.as_tensor(frame, device=device, dtype=torch.float32).permute(2, 0, 1) / 255.0
    t = F.interpolate(t.unsqueeze(0), size=(H, W), mode="bilinear", align_corners=False)[0]
    self._vid_cache = (idx, t)
    return t

def _guide_image(self, H, W, device):
    if getattr(self, "_img_guide", None) is None or self._img_guide.shape[-2:] != (H, W):
        from PIL import Image
        from pytti import fetch
        im = Image.open(fetch(SHP["guide_path"])).convert("RGB")
        t = TF.to_tensor(im).to(device)
        self._img_guide = F.interpolate(t.unsqueeze(0), size=(H, W), mode="bilinear",
                                        align_corners=False)[0]
    return self._img_guide

def _guide_field(self, H, W, step, device):
    src = SHP["guide_src"]
    if src == "reaction_diffusion":
        if getattr(self, "_rd_U", None) is None or self._rd_U.shape != _rd_grid(H, W):
            _rd_init(self, H, W, device)
        U, V = self._rd_U, self._rd_V
        f, k = SHP["rd_feed"], SHP["rd_kill"]
        for _ in range(SHP["rd_iters"]):
            uvv = U * V * V
            U = U + (0.16 * _laplace(U) - uvv + f * (1.0 - U))
            V = V + (0.08 * _laplace(V) + uvv - (f + k) * V)
            U.clamp_(0, 1); V.clamp_(0, 1)
        self._rd_U, self._rd_V = U, V
        g = F.interpolate(V.view(1, 1, *V.shape), size=(H, W), mode="bilinear",
                          align_corners=False)[0, 0]
    elif src == "video":
        g = _to_luma(_guide_video(self, H, W, step, device))
    else:
        g = _to_luma(_guide_image(self, H, W, device))
    g = _to_luma(g)
    if SHP["ginv"]:
        g = 1.0 - g
    if SHP["gblur"] > 0:
        r = SHP["gblur"]; kk = 2 * r + 1
        g = F.avg_pool2d(F.pad(g.view(1, 1, H, W), (r, r, r, r), mode="reflect"), kk, stride=1)[0, 0]
    return g

# ---- the full preconditioner: spectral envelope (+band) in Fourier space, then spatial guide -----
def _env_for(self, H, W, step, device):
    # The frequency envelope is CONSTANT unless the shape is animated. So we cache the radial+band
    # static part (built once) and the full envelope (rebuilt only when animated), leaving just the
    # two unavoidable gradient FFTs per backward. id(SHP) in the key auto-invalidates on cell re-run.
    animated = SHP["enabled"] and SHP["mix"] > 0 and SHP["animate"] and SHP["shape"] in _ANIMATED
    tok = (id(SHP), int(H), int(W))
    st = getattr(self, "_sp_static", None)
    if st is None or st[0] != tok:
        fy = torch.fft.fftfreq(H, device=device).abs().view(-1, 1)
        fx = torch.fft.rfftfreq(W, device=device).abs().view(1, -1)
        f = torch.sqrt(fy * fy + fx * fx); f[0, 0] = 1.0
        if SHP["enabled"]:
            radial = 1.0 / (f ** SHP["alpha"]); radial = radial / radial.mean()
        else:
            radial = torch.ones_like(f)
        band = _band_window(f) if SHP["band"] else None
        st = (tok, f, radial, band); self._sp_static = st
        self._sp_env = None
    _, f, radial, band = st
    if not animated:
        ce = getattr(self, "_sp_env", None)
        if ce is not None and ce[0] == tok:
            return ce[1]
    if SHP["enabled"] and SHP["mix"] > 0:
        S = _shape_field(H, W, step, device)
        Se = torch.fft.rfft2(S.float()).abs(); Se = Se / (Se.mean() + 1e-8)
        env = (1 - SHP["mix"]) * radial + SHP["mix"] * (Se ** SHP["alpha"])
    elif SHP["enabled"]:
        env = radial
    else:
        env = torch.ones_like(f)                       # band-only: flat base, just isolate the band
    if band is not None:
        env = env * band
    env = env / (env.mean() + 1e-8)                     # preserve overall gradient energy
    if SHP["detail_floor"] > 0:                         # don't let high freqs be suppressed -> zoom keeps re-detailing
        env = env.clamp(min=SHP["detail_floor"]); env = env / (env.mean() + 1e-8)
    if not animated:
        self._sp_env = (tok, env)
    return env

def _precondition(self, g, step):
    H, W = g.shape[-2], g.shape[-1]
    if SHP["enabled"] or SHP["band"]:
        env = _env_for(self, H, W, step, g.device)     # cached -> only 2 gradient FFTs below
        g_norm0 = g.norm()
        G = torch.fft.rfft2(g.float())
        g = torch.fft.irfft2(G * env, s=(H, W)).to(g.dtype)
        g = g * (g_norm0 / (g.norm() + 1e-12))         # norm-preserving: no magnitude runaway -> black collapse
    if SHP["guide"]:
        gf = _guide_field(self, H, W, step, g.device).to(g.dtype)
        if SHP["guide_mode"] == "attract":
            g = g + SHP["gstr"] * (self.value.detach() - gf)   # pull luma toward the guide
        else:
            s = float(max(0.0, min(1.0, SHP["gstr"])))
            g = g * (1.0 - s + s * gf)                         # gate structure to bright regions
    return g

# ---- chain an extra gradient hook onto PixelImage without disturbing 2.17's hook ----------------
try:
    if not getattr(PixelImage, "_shape_spectral_installed", False):
        PixelImage._pre_shape_init = PixelImage.__init__
        def _shape_init(self, *a, **k):
            PixelImage._pre_shape_init(self, *a, **k)
            def hook(grad):
                if not (SHP["enabled"] or SHP["band"] or SHP["guide"]):
                    return grad
                try:
                    step = getattr(self, "_sp_step", 0); self._sp_step = step + 1
                    return _precondition(self, grad, step)
                except Exception:
                    traceback.print_exc()
                    return grad
            self.value.register_hook(hook)
        PixelImage.__init__ = _shape_init
        PixelImage._shape_spectral_installed = True
    print("[ok] shape-spectral hook installed (shape:", SHP["enabled"], "| band:", SHP["band"],
          "->", f"{SHP['smin_px']:.0f}-{SHP['smax_px']:.0f}px", "| guide:", SHP["guide"],
          "->", SHP["guide_src"], SHP["guide_mode"], ")")
except Exception:
    traceback.print_exc()

# ---- quick preview: shape field + guide field (if enabled) --------------------------------------
if SHP["enabled"] or SHP["guide"]:
    try:
        import matplotlib.pyplot as plt
        cols = 1 + int(SHP["guide"])
        fig, axs = plt.subplots(1, cols, figsize=(2.4 * cols, 2.4)); axs = [axs] if cols == 1 else axs
        axs[0].imshow(_shape_field(160, 160, 0, torch.device("cpu")).numpy(), cmap="magma")
        axs[0].set_title(f"{SHP['shape']}"); axs[0].axis("off")
        if SHP["guide"]:
            class _Dummy: pass
            d = _Dummy(); d.value = torch.zeros(160, 160)
            try:
                gf = _guide_field(d, 160, 160, 0, torch.device("cpu")).numpy()
                axs[1].imshow(gf, cmap="magma"); axs[1].set_title(f"guide:{SHP['guide_src']}")
            except Exception:
                axs[1].set_title("guide: (preview failed)")
            axs[1].axis("off")
        plt.show()
    except Exception:
        traceback.print_exc()


In [ ]:
#@title 2.22  Frame-coherence optimizer  (A/B)  { form-width: "320px" }
#@markdown A zoom makes consecutive frames nearly the SAME optimization problem (predictor = the
#@markdown zoom warp, corrector = a few CLIP steps). These levers cut how many BACKWARD passes you
#@markdown spend per frame -- no generator, no training. All default OFF; flip ONE at a time and
#@markdown watch the profiler + the printed `backward/frame`. Run AFTER 2.1 / 2.20 / 2.21, BEFORE 2.3.

import math, builtins, traceback
import torch
import torch.nn.functional as F
from pytti.ImageGuide import DirectImageGuide
from pytti.Notebook import tqdm
import pytti.Transforms as _T

enable_frame_opt   = False  #@param{type:"boolean"}
#@markdown #### Adaptive corrector -- stop optimizing a frame once its loss plateaus (cap = steps_per_frame)
adaptive_corrector = True   #@param{type:"boolean"}
adaptive_min_steps = 3      #@param{type:"number"}
adaptive_patience  = 2      #@param{type:"number"}
adaptive_tol       = 0.002  #@param{type:"number"}
#@markdown #### Warp reuse -- on "reuse" frames, skip CLIP entirely: warp the cached gradient by the
#@markdown EXACT zoom/depth warp field (captured from Transforms.apply_grid, no recompute) and step.
#@markdown warp_recompute_every=N -> 1 real (CLIP) frame then N-1 cheap warped-reuse frames.
warp_reuse           = False #@param{type:"boolean"}
warp_recompute_every = 2     #@param{type:"number"}
#@markdown #### Lazy gradient -- within a real frame, replace some CLIP steps with reuse of last grad
reuse_grad         = False  #@param{type:"boolean"}
reuse_ratio        = 1      #@param{type:"number"}
#@markdown #### Coast cutouts -- fewer cutouts on refinement steps (first step of each frame is full)
coast_cutn         = False  #@param{type:"boolean"}
coast_cutn_value   = 16     #@param{type:"number"}
verbose            = True   #@param{type:"boolean"}

FOPT = dict(enabled=enable_frame_opt,
            adaptive=adaptive_corrector, min_steps=max(1, int(adaptive_min_steps)),
            patience=max(1, int(adaptive_patience)), tol=float(adaptive_tol),
            warp=warp_reuse, warp_every=max(2, int(warp_recompute_every)),
            reuse=reuse_grad, reuse_ratio=max(1, int(reuse_ratio)),
            coast=coast_cutn, coast_cutn=max(1, int(coast_cutn_value)),
            verbose=verbose)
builtins.FOPT = FOPT
if not hasattr(builtins, "_WARP_Q"):
    builtins._WARP_Q = []

# ---- capture the EXACT warp field: every 2D/3D/flow warp funnels through Transforms.apply_grid ----
# (zoom_3d -> render_image_3d -> apply_grid; grid = uv - depth_offset). Module-global patch, so the
# in-Transforms callers pick it up. We just stash the grid; no extra computation.
try:
    if not getattr(_T, "_apply_grid_captured", False):
        _T._orig_apply_grid = _T.apply_grid
        def _apply_grid_cap(tensor, grid, border_mode, sampling_mode):
            try:
                builtins._WARP_Q.append(grid.detach())
            except Exception:
                pass
            return _T._orig_apply_grid(tensor, grid, border_mode, sampling_mode)
        _T.apply_grid = _apply_grid_cap
        _T._apply_grid_captured = True
except Exception:
    traceback.print_exc()

def _cache_grads(guide):
    guide._wr_cache = [(p.grad.detach().clone() if p.grad is not None else None)
                       for p in guide.image_rep.parameters()]
    builtins._WARP_Q = []        # warps AFTER this real grad get applied to this cache

def _warp_reuse_step(guide, grids):
    cache = getattr(guide, "_wr_cache", None)
    params = list(guide.image_rep.parameters())
    if not cache or len(cache) != len(params):
        return False
    with torch.no_grad():
        for p, gc in zip(params, cache):
            if gc is None:
                p.grad = None; continue
            g = gc
            for grid in grids:                                  # apply each captured warp, in order
                Hg, Wg = int(grid.shape[1]), int(grid.shape[2])
                if g.shape[-2:] == (Hg, Wg):                    # spatial params (value [H,W], tensor [n,H,W])
                    shp = g.shape
                    x = g.reshape(1, -1, Hg, Wg).float()
                    x = F.grid_sample(x, grid.to(x.device).float(), mode="bilinear",
                                      padding_mode="border", align_corners=True)
                    g = x.reshape(shp).to(gc.dtype)
            p.grad = g.clone()                                  # non-spatial (palette) reused as-is
    guide.optimizer.step(); guide.image_rep.update()
    return True

try:
    if not hasattr(DirectImageGuide, "_orig_run_steps_fopt"):
        DirectImageGuide._orig_run_steps_fopt = DirectImageGuide.run_steps

    def _run_steps_fopt(self, n_steps, prompts, interp_prompts, loss_augs,
                        stop=-math.inf, interp_steps=0, i_offset=0, skipped_steps=0):
        if not FOPT["enabled"]:
            return DirectImageGuide._orig_run_steps_fopt(
                self, n_steps, prompts, interp_prompts, loss_augs,
                stop=stop, interp_steps=interp_steps, i_offset=i_offset, skipped_steps=skipped_steps)

        g = globals()
        p = g.get("params", getattr(builtins, "params", None))
        pre = int(getattr(p, "pre_animation_steps", 0)) if p is not None else 0
        spf = max(1, int(getattr(p, "steps_per_frame", 1))) if p is not None else 1
        mot = g.get("MOT", getattr(builtins, "MOT", None))
        sec_motion = g.get("_apply_secondary_motion", None)
        sec_boundary = g.get("_is_frame_boundary", None)
        emb = self.embedder
        base_cutn = getattr(emb, "cutn", None) if emb is not None else None

        frame_done = False; prev = None; in_frame = 0; flat = 0; step_in_frame = 0
        warp_frame = False
        real = 0; reused = 0; warped = 0; frames = 0; i = -1
        for i in tqdm(range(n_steps)):
            gi = i + i_offset
            boundary = (gi >= pre) and ((gi - pre) % spf == 0)
            if boundary:
                frames += 1
                fidx = (gi - pre) // spf
                have_cache = getattr(self, "_wr_cache", None) is not None
                warp_frame = FOPT["warp"] and have_cache and (fidx % FOPT["warp_every"] != 0)
                frame_done = False; prev = None; in_frame = 0; flat = 0; step_in_frame = 0

            self.update(gi, i + skipped_steps)                       # primary zoom/3D warp -> captured here
            if mot and mot.get("enabled") and sec_motion and sec_boundary:
                try:
                    if sec_boundary(gi):
                        sec_motion(self.image_rep)
                except Exception:
                    traceback.print_exc()

            if frame_done:
                continue

            # ---- warp-reuse frame: one warped-gradient step (no CLIP), then coast ----
            if warp_frame:
                if _warp_reuse_step(self, list(builtins._WARP_Q)):
                    warped += 1
                else:                                               # cache missing -> fall back to real
                    self.train(i + skipped_steps, prompts, interp_prompts, loss_augs, interp_steps=interp_steps)
                    _cache_grads(self); real += 1
                frame_done = True
                continue

            # ---- real frame: normal / adaptive / lazy steps ----
            do_real = True
            if FOPT["reuse"] and step_in_frame > 0:
                do_real = (step_in_frame % (FOPT["reuse_ratio"] + 1) == 0)
            step_in_frame += 1

            if do_real:
                if FOPT["coast"] and base_cutn is not None:
                    emb.cutn = base_cutn if in_frame == 0 else int(FOPT["coast_cutn"])
                losses = self.train(i + skipped_steps, prompts, interp_prompts, loss_augs,
                                    interp_steps=interp_steps)
                if base_cutn is not None:
                    emb.cutn = base_cutn
                _cache_grads(self)                                  # keep newest real grad for warp reuse
                real += 1; in_frame += 1
                l = losses["TOTAL"]
                if FOPT["adaptive"] and prev is not None and in_frame >= FOPT["min_steps"]:
                    if (prev - l) < FOPT["tol"] * max(1e-8, abs(prev)):
                        flat += 1
                        if flat >= FOPT["patience"]:
                            frame_done = True
                    else:
                        flat = 0
                prev = l
                if l <= stop:
                    break
            else:
                self.optimizer.step(); self.image_rep.update(); reused += 1

        if base_cutn is not None:
            emb.cutn = base_cutn
        if FOPT["verbose"] and frames > 0:
            print(f"[frame-opt] real {real} + lazy {reused} + warp {warped} over ~{frames} frames "
                  f"= {real / max(1, frames):.2f} CLIP-backward/frame (baseline {spf})")
        return i + 1

    DirectImageGuide.run_steps = _run_steps_fopt
    print("[ok] frame-coherence optimizer installed (enabled:", FOPT["enabled"],
          "| adaptive:", FOPT["adaptive"], "| warp:", FOPT["warp"], "every", FOPT["warp_every"],
          "| lazy:", FOPT["reuse"], "| coast:", FOPT["coast"], ")")
except Exception:
    traceback.print_exc()


In [ ]:
#@title 2.3 Run it!
#@markdown pytti is 1000% percent better code than VQLIPSE, so have a look at the code.
#@markdown You just might understand what's going on.

import os
import sys
import gc
import torch
import warnings
import numpy as np
import subprocess
import re
import math
import json
from os.path import exists as path_exists
from IPython import display
from PIL import Image, ImageEnhance
from torchvision.transforms import functional as TF

# Append AdaBins to system path
sys.path.append('./AdaBins')

# Check if the drive is mounted and set the working directory
drive_mounted = path_exists('/content/drive/MyDrive/pytti_test')
if drive_mounted:
    %cd /content/drive/MyDrive/pytti_test

try:
    from pytti.Notebook import *
    from pytti import Perceptor
except ModuleNotFoundError as e:
    error_msg = 'ERROR: please run setup (step 1.3).'
    if not drive_mounted:
        error_msg = 'WARNING: drive is not mounted.\n' + error_msg
    raise RuntimeError(error_msg) from e

print("Loading pytti...")
from pytti.Image import PixelImage, RGBImage, VQGANImage, MultiResImage, LimitedPaletteMultiResImage, GNCAImage
from pytti.ImageGuide import DirectImageGuide, EnhancedImageGuide
from pytti.Perceptor.Embedder import HDMultiClipEmbedder
from pytti.Perceptor.Prompt import parse_prompt
from pytti.LossAug import TVLoss, OpticalFlowLoss, TargetFlowLoss
from pytti.Transforms import zoom_2d, zoom_3d
from pytti import *
from pytti.LossAug.DepthLoss import init_AdaBins
print("pytti loaded.")

# Display settings
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set()
plt.style.use('bmh')
pd.options.display.max_columns = None
pd.options.display.width = 175

# Initialize variables
latest = -1
#@markdown check `batch_mode` to run batch settings
batch_mode = False #@param{type:"boolean"}

if batch_mode:
    try:
        batch_list
    except NameError:
        raise RuntimeError("ERROR: no batch settings. Please run 'batch settings' cell at the bottom of the page to use batch mode.")
else:
    try:
        params
    except NameError:
        raise RuntimeError("ERROR: no parameters. Please run parameters (step 2.1).")

#@markdown check `restore` to restore from a previous run
restore = False#@param{type:"boolean"}
#@markdown check `reencode` if you are restoring with a modified image or modified image settings
reencode = False#@param{type:"boolean"}
#@markdown which run to restore
restore_run =  -1#@param{type:"raw"}

if restore and restore_run == latest:
    _, restore_run = get_last_file(
        f'backup/{params.file_namespace}',
        f'^(?P<pre>{re.escape(params.file_namespace)}\\(?)(?P<index>\\d*)(?P<post>\\)?_\\d+\\.bak)$'
    )

def do_run():
    clear_rotoscopers()
    vram_profiling(params.approximate_vram_usage)
    reset_vram_usage()

    #@markdown which frame to restore from
    restore_frame =  -1#@param{type:"raw"}
    # Set seed for deterministic RNG
    if params.seed is not None:
        torch.manual_seed(params.seed)

    # Load CLIP and create embedder
    load_clip(params)
    embedder = HDMultiClipEmbedder(
        cutn=params.cutouts,
        cut_pow=params.cut_pow,
        padding=params.cutout_border,
        border_mode=params.border_mode
    )

    # Load scenes
    with vram_usage_mode('Text Prompts'):
        print('Loading prompts...')
        prompts = []
        for stage in params.scenes.split('||'):
            if stage:
                scene_prompts = []
                for p in (params.scene_prefix + stage + params.scene_suffix).strip().split('|'):
                    p = p.strip()
                    if p:
                        scene_prompts.append(parse_prompt(embedder, p))
                prompts.append(scene_prompts)
        print('Prompts loaded.')

    # Load initial image if provided
    init_image_pil = None
    if params.init_image:
        init_image_pil = Image.open(fetch(params.init_image)).convert('RGB')
        init_size = init_image_pil.size
        if params.width == -1:
            params.width = int(params.height * init_size[0] / init_size[1])
        if params.height == -1:
            params.height = int(params.width * init_size[1] / init_size[0])

    # Video source handling
    video_frames = None
    if params.animation_mode == "Video Source":
        print(f'Loading video from {params.video_path}...')
        video_frames = get_frames(params.video_path)
        params.pre_animation_steps = max(params.steps_per_frame, params.pre_animation_steps)
        if not init_image_pil:
            init_image_pil = Image.fromarray(video_frames.get_data(0)).convert('RGB')
            init_size = init_image_pil.size
            if params.width == -1:
                params.width = int(params.height * init_size[0] / init_size[1])
            if params.height == -1:
                params.height = int(params.width * init_size[1] / init_size[0])

    # Initialize image model
    if params.image_model == "Limited Palette":
        img = PixelImage(*format_params(params,
            'width', 'height', 'pixel_size', 'palette_size', 'palettes', 'gamma',
            'hdr_weight', 'palette_normalization_weight'))
        img.encode_random(random_pallet=params.random_initial_palette)
        if params.target_palette.strip():
            img.set_pallet_target(Image.open(fetch(params.target_palette)).convert('RGB'))
        else:
            img.lock_pallet(params.lock_palette)
    elif params.image_model == "Unlimited Palette":
        img = RGBImage(params.width, params.height, params.pixel_size)
        img.encode_random()
    elif params.image_model == "VQGAN":
        VQGANImage.init_vqgan(params.vqgan_model)
        img = VQGANImage(params.width, params.height, params.pixel_size)
        img.encode_random()
    elif params.image_model == "MultiRes(DAS)":
        img = MultiResImage(params.width, params.height,pixel_format='RGB', scales=[1], gamma=params.gamma, init='random')
        img.encode_random()
    elif params.image_model == "MultiResLimitedPalette":
        img = LimitedPaletteMultiResImage(params.width, params.height, palette_size=params.palette_size,
                                          scales=[1], gamma=params.gamma)
        #learning_rate = params.learning_rate
        img.encode_random(random_palette=params.random_initial_palette)
        if params.target_palette.strip():
            img.set_palette_target(Image.open(fetch(params.target_palette)).convert('RGB'))
        else:
            img.lock_pallet(params.lock_palette)
    elif params.image_model == "GNCA":
        img = GNCAImage(
            params.width, params.height,
            scale=params.pixel_size,
            pallet_size=params.palette_size,
            n_pallets=params.palettes,
            gamma=params.gamma,
            hdr_weight=params.hdr_weight,
            norm_weight=params.palette_normalization_weight
        )

        # Initialize either random or from target image
        if params.random_initial_palette:
            img.encode_random(random_pallet=True)
        else:
            img.reset_state()

        # Set up palette if target specified
        if params.target_palette.strip():
            img.set_pallet_target(Image.open(fetch(params.target_palette)).convert('RGB'))
        else:
            img.lock_pallet(params.lock_palette)

        # Set animation parameters
        img.set_steps_per_update(1)  # Start with slow growth
        img.set_update_mode('grow')  # 'grow' or 'none'

        # If init image is provided, encode it
        if params.init_image.strip():
            target_img = Image.open(fetch(params.init_image)).convert('RGB')
            img.encode_image(target_img, smart_encode=True)

    else:
        raise ValueError(f'Unknown image model: {params.image_model}')


    # Set up loss augmentations
    loss_augs = []
    if init_image_pil:
        if not restore:
            print("Encoding initial image...")
            img.encode_image(init_image_pil, smart_encode=True)
            print("Initial image encoded.")
            display.display(img.decode_image())

        # Initialize image prompt
        init_augs = []
        if params.direct_init_weight not in ['', '0']:
            init_aug = build_loss(
                'direct_init_weight', params.direct_init_weight,
                f'init image ({params.init_image})', img, init_image_pil
            )
            init_augs.append(init_aug)
        loss_augs.extend(init_augs)

        # Semantic initial prompt
        semantic_init_prompt = None
        if params.semantic_init_weight not in ['', '0']:
            semantic_init_prompt = parse_prompt(
                embedder,
                f"init image [{params.init_image}]:{params.semantic_init_weight}",
                init_image_pil
            )
            prompts[0].append(semantic_init_prompt)
    else:
        init_augs, semantic_init_prompt = [], None

    # Other image prompts
    for p in params.direct_image_prompts.split('|'):
        p = p.strip()
        if p:
            loss_augs.append(type(img).get_preferred_loss().TargetImage(p, img.image_shape, is_path=True))

    # Stabilization augmentations
    stabilization_augs = []
    for weight_param in ['direct_stabilization_weight', 'depth_stabilization_weight', 'edge_stabilization_weight']:
        weight = params[weight_param]
        if weight not in ['', '0']:
            stabilization_augs.append(build_loss(weight_param, weight, 'stabilization', img, init_image_pil))
    loss_augs.extend(stabilization_augs)

    # Semantic stabilization
    last_frame_semantic = None
    if params.semantic_stabilization_weight not in ['', '0']:
        last_frame_semantic = parse_prompt(
            embedder,
            f"stabilization:{params.semantic_stabilization_weight}",
            init_image_pil if init_image_pil else img.decode_image()
        )
        last_frame_semantic.set_enabled(bool(init_image_pil))
        for scene in prompts:
            scene.append(last_frame_semantic)

    # Optical flow
    optical_flows = []
    if params.animation_mode == 'Video Source':
        flow_weight = params.flow_stabilization_weight or '0'
        optical_flows = [
            OpticalFlowLoss.TargetImage(
                f"optical flow stabilization (frame {-2**i}):{flow_weight}", img.image_shape
            )
            for i in range(params.flow_long_term_samples + 1)
        ]
        for optical_flow in optical_flows:
            optical_flow.set_enabled(False)
        loss_augs.extend(optical_flows)
    elif params.animation_mode == '3D' and params.flow_stabilization_weight not in ['', '0']:
        optical_flow = TargetFlowLoss.TargetImage(
            f"optical flow stabilization:{params.flow_stabilization_weight}", img.image_shape
        )
        optical_flow.set_enabled(False)
        optical_flows.append(optical_flow)
        loss_augs.extend(optical_flows)

    # Smoothing
    if params.smoothing_weight != 0:
        loss_augs.append(TVLoss(weight=params.smoothing_weight))

    # Set up file paths
    output_dir = f'images_out/{params.file_namespace}'
    backup_dir = f'backup/{params.file_namespace}'
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(backup_dir, exist_ok=True)

    # Handle restore
    if restore:
        base_name = params.file_namespace if restore_run == 0 else f'{params.file_namespace}({restore_run})'
    elif not params.allow_overwrite:
        _, idx = get_next_file(
            output_dir,
            f'^(?P<pre>{re.escape(params.file_namespace)}\\(?)(?P<index>\\d*)(?P<post>\\)?_1\\.png)$',
            [f"{params.file_namespace}_1.png", f"{params.file_namespace}(1)_1.png"]
        )
        base_name = params.file_namespace if idx == 0 else f'{params.file_namespace}({idx})'
    else:
        base_name = params.file_namespace

    # Restore from backup if necessary
    if restore:
        if not reencode:
            if restore_frame == latest:
                filename, restore_frame = get_last_file(
                    backup_dir, f'^(?P<pre>{re.escape(base_name)}_)(?P<index>\\d*)(?P<post>\\.bak)$'
                )
            else:
                filename = f'{base_name}_{restore_frame}.bak'
            print("Restoring from", filename)
            img.load_state_dict(torch.load(f'{backup_dir}/{filename}'))
        else:
            if restore_frame == latest:
                filename, restore_frame = get_last_file(
                    output_dir, f'^(?P<pre>{re.escape(base_name)}_)(?P<index>\\d*)(?P<post>\\.png)$'
                )
            else:
                filename = f'{base_name}_{restore_frame}.png'
            print("Restoring from", filename)
            img.encode_image(Image.open(f'{output_dir}/{filename}').convert('RGB'))
        i = restore_frame * params.save_every
    else:
        i = 0

    # Initialize graphs
    fig, axs = None, None
    if params.show_graphs:
        fig, axs = plt.subplots(4, 1, figsize=(21, 13))
        axs = axs.flatten()

    # Create the main model object
    model = DirectImageGuide(img, embedder, lr=params.learning_rate)
    """
    model = EnhancedImageGuide(
            img,
            embedder,
            optimizer_name="adamw",# Try 'adam', 'adamw', 'radam', or 'lookahead'
            adaptive_weights= True,
            lr=params.learning_rate,
            weight_update_freq=15,
            weight_scale_factor=1    )
    """
    # Define update function
    def update(i, stage_i):
        # Display updates
        if params.clear_every > 0 and i % params.clear_every == 0:
            display.clear_output()
        if params.display_every > 0 and i % params.display_every == 0:
            print(f"Step {i} losses:")
            if model.dataframe:
                print(model.dataframe[0].iloc[-1])
            if params.approximate_vram_usage:
                print("VRAM Usage:")
                print_vram_usage()

            display_width = int(img.image_shape[0] * params.display_scale)
            display_height = int(img.image_shape[1] * params.display_scale)
            im = img.decode_image()
            if stage_i > 0 and params.show_graphs:
                model.plot_losses(axs)
                side_by_side = make_hbox(im.resize((display_width, display_height), Image.LANCZOS), fig)
                display.display(side_by_side)
            else:
                display.display(im.resize((display_width, display_height), Image.LANCZOS))
            if params.show_palette and (isinstance(img, PixelImage) or isinstance(img, LimitedPaletteMultiResImage)):
                print('Palette:')
                display.display(img.render_pallet())

        # Save checkpoints
        if i > 0 and params.save_every > 0 and i % params.save_every == 0:
            im = img.decode_image()
            n = i // params.save_every
            im.save(f"{output_dir}/{base_name}_{n}.png")
            if params.backups > 0:
                torch.save(img.state_dict(), f"{backup_dir}/{base_name}_{n}.bak")
                old_backup = f"{backup_dir}/{base_name}_{n - params.backups}.bak"
                if n > params.backups and os.path.exists(old_backup):
                    os.remove(old_backup)

        # Animation
        t = (i - params.pre_animation_steps) / (params.steps_per_frame * params.frames_per_second)
        set_t(t)
        if i >= params.pre_animation_steps and (i - params.pre_animation_steps) % params.steps_per_frame == 0:
            print(f"Time: {t:.4f} seconds")
            update_rotoscopers(((i - params.pre_animation_steps) // params.steps_per_frame + 1) * params.frame_stride)
            if params.reset_lr_each_frame:
                model.set_optim(None)
            next_step_pil = None

            if params.animation_mode == "2D":
                tx = parametric_eval(params.translate_x)
                ty = parametric_eval(params.translate_y)
                theta = parametric_eval(params.rotate_2d)
                zx = parametric_eval(params.zoom_x_2d)
                zy = parametric_eval(params.zoom_y_2d)
                next_step_pil = zoom_2d(
                    img, (tx, ty), (zx, zy), theta,
                    border_mode=params.infill_mode,
                    sampling_mode=params.sampling_mode
                )
            elif params.animation_mode == "3D":
                im = img.decode_image()
                with vram_usage_mode('Optical Flow Loss'):
                    flow, next_step_pil = zoom_3d(
                        img, (params.translate_x, params.translate_y, params.translate_z_3d),
                        params.rotate_3d, params.field_of_view, params.near_plane, params.far_plane,
                        border_mode=params.infill_mode, sampling_mode=params.sampling_mode,
                        stabilize=params.lock_camera
                    )
                    freeze_vram_usage()
                for optical_flow in optical_flows:
                    optical_flow.set_last_step(im)
                    optical_flow.set_target_flow(flow)
                    optical_flow.set_enabled(True)
            elif params.animation_mode == "Video Source" and video_frames:
                frame_n = min(
                    (i - params.pre_animation_steps) * params.frame_stride // params.steps_per_frame,
                    video_frames.count_frames() - 1
                )
                next_frame_n = min(frame_n + params.frame_stride, video_frames.count_frames() - 1)
                next_step_pil = Image.fromarray(video_frames.get_data(next_frame_n)).convert('RGB').resize(
                    img.image_shape, Image.LANCZOS
                )
                for j, optical_flow in enumerate(optical_flows):
                    old_frame_n = frame_n - (2 ** j - 1) * params.frame_stride
                    save_n = i // params.save_every - (2 ** j - 1)
                    if old_frame_n < 0 or save_n < 1:
                        break
                    current_step_pil = Image.fromarray(video_frames.get_data(old_frame_n)).convert('RGB').resize(
                        img.image_shape, Image.LANCZOS
                    )
                    filename = None if j == 0 else f"{backup_dir}/{base_name}_{save_n}.bak"
                    flow_im, mask_tensor = optical_flow.set_flow(
                        current_step_pil, next_step_pil, img, filename,
                        params.infill_mode, params.sampling_mode
                    )
                    optical_flow.set_enabled(True)
                    if j == 0:
                        mask_accum = mask_tensor.detach()
                        valid = mask_tensor.mean()
                        print("Valid pixels:", valid.item())
                        if params.reencode_each_frame or valid < .03:
                            if isinstance(img, PixelImage) or isinstance(img, LimitedPaletteMultiResImage) and valid >= .03:
                                img.lock_pallet()
                                img.encode_image(next_step_pil, smart_encode=False)
                                img.lock_pallet(params.lock_palette)
                            else:
                                img.encode_image(next_step_pil)
                            reencoded = True
                        else:
                            reencoded = False
                    else:
                        with torch.no_grad():
                            optical_flow.set_mask((mask_tensor - mask_accum).clamp(0, 1))
                            mask_accum.add_(mask_tensor)

            if params.animation_mode != 'off' and next_step_pil:
                for aug in stabilization_augs:
                    aug.set_comp(next_step_pil)
                    aug.set_enabled(True)
                if last_frame_semantic:
                    last_frame_semantic.set_image(embedder, next_step_pil)
                    last_frame_semantic.set_enabled(True)
                for aug in init_augs:
                    aug.set_enabled(False)
                if semantic_init_prompt:
                    semantic_init_prompt.set_enabled(False)

    model.update = update

    # Save settings
    settings_path = f"{output_dir}/{base_name}_settings.txt"
    print(f"Settings saved to {settings_path}")
    save_settings(params, settings_path)

    # Run steps
    skip_prompts = i // params.steps_per_scene
    skip_steps = i % params.steps_per_scene
    last_scene = prompts[0] if skip_prompts == 0 else prompts[skip_prompts - 1]

    for scene in prompts[skip_prompts:]:
        print("Running prompt:", ' | '.join(map(str, scene)))
        steps_remaining = params.steps_per_scene - skip_steps
        i += model.run_steps(
            steps_remaining, scene, last_scene, loss_augs,
            interp_steps=params.interpolation_steps,
            i_offset=i, skipped_steps=skip_steps
        )
        skip_steps = 0
        model.clear_dataframe()
        last_scene = scene

    if fig:
        plt.close(fig)
        del fig, axs

try:
    gc.collect()
    torch.cuda.empty_cache()
    if batch_mode:
        if restore:
            settings_list = batch_list[restore_run:]
        else:
            settings_list = batch_list
            namespace = batch_list[0]['file_namespace']
            os.makedirs(f'images_out/{namespace}', exist_ok=True)
            save_batch(batch_list, f'images_out/{namespace}/{namespace}_batch_settings.txt')
            print(f"Batch settings saved to images_out/{namespace}/{namespace}_batch_settings.txt")
        for settings in settings_list:
            setting_string = json.dumps(settings)
            print("SETTINGS:")
            print(setting_string)
            params = load_settings(setting_string)
            if params.animation_mode == '3D':
                init_AdaBins()
            params.allow_overwrite = False
            do_run()
            restore = False
            reencode = False
            gc.collect()
            torch.cuda.empty_cache()
    else:
        if params.animation_mode == '3D':
            init_AdaBins()
        do_run()
        print("Complete.")
        gc.collect()
        torch.cuda.empty_cache()
except KeyboardInterrupt:
    pass
except RuntimeError as e:
    print_vram_usage()
    raise e

In [ ]:
#@title  Reset cache & empty VRAM
import gc, torch

def _gb(x): return x / 1024**3

if torch.cuda.is_available():
    print(f"Before:  allocated {_gb(torch.cuda.memory_allocated()):.2f} GB | "
          f"reserved {_gb(torch.cuda.memory_reserved()):.2f} GB")

# drop big globals if they exist
for _name in ["model", "img", "embedder", "z", "prompts", "loss_augs",
              "image_embeds", "offsets", "sizes", "next_step_pil", "fig", "axs"]:
    if _name in dir():
        try: del globals()[_name]
        except Exception: pass

# free the frozen CLIP / VQGAN backbones if pytti is loaded
try:
    import pytti.Perceptor as _P
    _P.free_clip()
except Exception: pass
try:
    from pytti.Image.VQGANImage import VQGANImage
    VQGANImage.free_vqgan()
except Exception: pass

# clear caches our experiment cells created
try:
    import builtins
    if hasattr(builtins, "_DG_CACHE"): builtins._DG_CACHE["depth"] = None
except Exception: pass
try:
    import pytti.Perceptor.Prompt as _PM
    _PM._SRC_EMB = None                 # directional-CLIP cached source embedding
except Exception: pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    print(f"After:   allocated {_gb(torch.cuda.memory_allocated()):.2f} GB | "
          f"reserved {_gb(torch.cuda.memory_reserved()):.2f} GB")
print("done.  (re-run 2.3 afterwards; CLIP/VQGAN were freed)")


# Step 3: Render video
You can dowload from the notebook, but it's faster to download from your drive.

In [ ]:
# prompt: put all png files from  '/home/administrator/pytti/images_out/{params.file_namespace}' to '/home/administrator/pytti/images_out/{params.file_namespace}/f'
#@title Stitch frames (Frames 2 Video)
import os
import cv2
from tqdm import tqdm
import natsort
import shutil
import glob

# project_name =  params.file_namespace
current_project = True #@param {type:"boolean"}
if current_project:
  project_name = params.file_namespace
else:
  project_name = "zeus"#@param {type:"string"}

# Assuming params.file_namespace is defined as in the provided code
source_dir = f'/home/administrator/pytti/images_out/{project_name}'
destination_dir = f'/home/administrator/pytti/images_out/{project_name}/f'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Find all PNG files in the source directory
png_files = glob.glob(os.path.join(source_dir, "*.png"))

# Move each PNG file to the destination directory
for file in png_files:
  shutil.move(file, destination_dir)

dir_path = destination_dir


# Get a list of files in the directory, sorted by last modified time
files = sorted(glob.glob(os.path.join(dir_path, '*')), key=lambda x: os.stat(x).st_mtime)

# Determine the number of digits needed for the filename
num_files = len(files)
num_digits = len(str(num_files))

# Rename the files
for i, file in enumerate(files, start=1):
    filename, file_extension = os.path.splitext(file)
    new_filename = f"{str(i).zfill(num_digits)}{file_extension}"
    os.rename(file, os.path.join(dir_path, new_filename))

#@markdown project name is the same as file_namespace
frames_path = destination_dir
#@markdown you have the option to process other projects in /images_out by disabling current project.
video_path = os.path.join(source_dir, "vid.mp4")

fps = 15 #@param {type:"integer"}

# Create a video given a folder of frames
def create_video(frames_folder, video_name, fps=30):
    images = [img for img in os.listdir(frames_folder) if img.endswith(".png")]
    images = natsort.natsorted(images)  # Use natural sorting
    if images == []: images = [img for img in os.listdir(frames_folder) if img.endswith(".png")]
    frame = cv2.imread(os.path.join(frames_folder, images[0]))
    height, width, layers = frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(video_name, fourcc, fps, (width, height))
    for image in tqdm(images):
        video.write(cv2.imread(os.path.join(frames_folder, image)))
    #cv2.destroyAllWindows()
    video.release()

create_video(frames_path, video_path, fps=fps)

In [ ]:
#@title Keyframe Generator
#@markdown feed the interpolated keyframes, pytti_kf, to your desired input.
keyframes="0:(60),100:(10),200:(40)"#@param{type:"string"}
pytti_kf = "(lambda builtins, fps, kf: kf[builtins[\"min\"](kf, key = lambda x: builtins[\"abs\"](x-(t*fps)//1))])([a for a in (1).__class__.__base__.__subclasses__() if a.__name__ == \"catch_warnings\"][0]()._module.__builtins__, 12, {"+ keyframes + "})"
pytti_kf

In [ ]:
#@title 3.2 (Optional) Interpolate video (Not Working rn)
force_install = False
cwd = os.getcwd()
exists = lambda path: os.path.exists(os.path.join(cwd,path))
if not exists("rife-ncnn-vulkan"):
  !git clone https://github.com/nihui/rife-ncnn-vulkan.git
  %cd rife-ncnn-vulkan
  wget https://github.com/nihui/rife-ncnn-vulkan/releases/download/20221029/rife-ncnn-vulkan-20221029-ubuntu.zip
  !unzip rife-ncnn-vulkan-20221029-ubuntu.zip

# Batch Setings
WARNING: If you use google colab (even with pro and pro+) GPUs for long enought google will throttle your account. Be careful with batch runs if you don't want to get kicked.

In [ ]:
#@title batch settings
from os.path import exists as path_exists
if path_exists('/content/drive/MyDrive/pytti_test'):
  %cd /content/drive/MyDrive/pytti_test
  drive_mounted = True
else:
  drive_mounted = False
try:
  from pytti.Notebook import change_tqdm_color, save_batch
except ModuleNotFoundError:
  if drive_mounted:
    raise RuntimeError('ERROR: please run setup (step 1).')
  else:
    raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1).')
change_tqdm_color()

try:
  import exrex, random, glob
except ModuleNotFoundError:
  if drive_mounted:
    raise RuntimeError('ERROR: please run setup (step 1).')
  else:
    raise RuntimeError('WARNING: drive is not mounted.\nERROR: please run setup (step 1).')
from numpy import arange
import itertools

def all_matches(s):
  return list(exrex.generate(s))

def dict_product(dictionary):
  return [dict(zip(dictionary, x)) for x in itertools.product(*dictionary.values())]

#these are used to make the defaults look pretty
model_default = None
random_seed = None

def define_parameters():
  locals_before = locals().copy()
  scenes = [""] #@param{type:"raw"}
  scene_prefix = ["all "," permutations "," are run "] #@param{type:"raw"}
  scene_suffix = [" that", " makes", " 27" ] #@param{type:"raw"}
  interpolation_steps = [0] #@param{type:"raw"}
  steps_per_scene = [500] #@param{type:"raw"}
  direct_image_prompts = [""] #@param{type:"raw"}
  init_image = [""] #@param{type:"raw"}
  direct_init_weight = [""] #@param{type:"raw"}
  semantic_init_weight = [""] #@param{type:"raw"}
  image_model = ["Limited Palette"] #@param{type:"raw"}
  width = [180] #@param{type:"raw"}
  height = [112] #@param{type:"raw"}
  pixel_size = [4] #@param{type:"raw"}
  smoothing_weight = [0.05] #@param{type:"raw"}
  vqgan_model = ["sflckr"] #@param{type:"raw"}
  random_initial_palette = [False] #@param{type:"raw"}
  palette_size = [9] #@param{type:"raw"}
  palettes = [8] #@param{type:"raw"}
  gamma = [1] #@param{type:"raw"}
  hdr_weight = [1.0] #@param{type:"raw"}
  palette_normalization_weight = [1.0] #@param{type:"raw"}
  show_palette = [False] #@param{type:"raw"}
  target_palette = [""] #@param{type:"raw"}
  lock_palette = [False] #@param{type:"raw"}
  animation_mode = ["off"] #@param{type:"raw"}
  sampling_mode = ["bicubic"] #@param{type:"raw"}
  infill_mode = ["wrap"] #@param{type:"raw"}
  pre_animation_steps = [100] #@param{type:"raw"}
  steps_per_frame = [50] #@param{type:"raw"}
  frames_per_second = [12] #@param{type:"raw"}
  direct_stabilization_weight = [""] #@param{type:"raw"}
  semantic_stabilization_weight = [""] #@param{type:"raw"}
  depth_stabilization_weight = [""] #@param{type:"raw"}
  edge_stabilization_weight = [""] #@param{type:"raw"}
  flow_stabilization_weight = [""] #@param{type:"raw"}
  video_path = [""] #@param{type:"raw"}
  frame_stride = [1] #@param{type:"raw"}
  reencode_each_frame = [True] #@param{type:"raw"}
  flow_long_term_samples = [0] #@param{type:"raw"}
  translate_x = ["0"] #@param{type:"raw"}
  translate_y = ["0"] #@param{type:"raw"}
  translate_z_3d = ["0"] #@param{type:"raw"}
  rotate_3d = ["[1,0,0,0]"] #@param{type:"raw"}
  rotate_2d = ["0"] #@param{type:"raw"}
  zoom_x_2d = ["0"] #@param{type:"raw"}
  zoom_y_2d = ["0"] #@param{type:"raw"}
  lock_camera = [True] #@param{type:"raw"}
  field_of_view = [60] #@param{type:"raw"}
  near_plane = [1] #@param{type:"raw"}
  far_plane = [10000] #@param{type:"raw"}
  file_namespace = ["Basic Batch"] #@param{type:"raw"}
  allow_overwrite = [False]
  display_every = [50] #@param{type:"raw"}
  clear_every = [0] #@param{type:"raw"}
  display_scale = [1] #@param{type:"raw"}
  save_every = [50] #@param{type:"raw"}
  backups = [2] #@param{type:"raw"}
  show_graphs = [False] #@param{type:"raw"}
  approximate_vram_usage = [False] #@param{type:"raw"}
  ViTB32 = [True] #@param{type:"raw"}
  ViTB16 = [False] #@param{type:"raw"}
  RN50 = [False] #@param{type:"raw"}
  RN50x4 = [False] #@param{type:"raw"}
  learning_rate = [None] #@param{type:"raw"}
  reset_lr_each_frame = [True] #@param{type:"raw"}
  seed = [None] #@param{type:"raw"}
  cutouts = [40] #@param{type:"raw"}
  cut_pow = [2] #@param{type:"raw"}
  cutout_border = [0.25] #@param{type:"raw"}
  border_mode = ["clamp"] #@param{type:"raw"}
  locals_after = locals().copy()
  for k in locals_before.keys():
    del locals_after[k]
  del locals_after['locals_before']
  return locals_after

param_dict = define_parameters()
batch_list = dict_product(param_dict)
namespace = batch_list[0]['file_namespace']
if glob.glob(f'images_out/{namespace}/*.png'):
  print(f"WARNING: images_out/{namespace} contains images. Batch indicies may not match filenames unless restoring.")